## notebook 1 - Analysis

In [1]:
# ==========================================
# n1-1 — Project & Pipeline Configuration (IMPROVED)
# ==========================================

import yaml
import random
import logging
from pathlib import Path

# ------------------------------------------
# a) Project metadata
# ------------------------------------------

DATASET_NAME = "260504-pbmc3k"
CONFIG_NAME = "pbmc"

# ------------------------------------------
# b) Paths & directories
# ------------------------------------------

# More robust base directory (safe for notebooks & scripts)

#BASE_DIR = Path("../").resolve()
BASE_DIR = Path.cwd().parents[0]

CONFIG_DIR = BASE_DIR / "configs"
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
DATA_DIR = BASE_DIR / "data" / DATASET_NAME
RESULTS_DIR = BASE_DIR / "results" / DATASET_NAME
FIG_DIR = BASE_DIR / "figures" / DATASET_NAME

# Create all required directories
for p in [RAW_DATA_DIR, DATA_DIR, RESULTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------
# c) Logging (PROFESSIONAL ADDITION)
# ------------------------------------------

LOG_PATH = RESULTS_DIR / "0-pipeline.log"

logging.basicConfig(
    filename=LOG_PATH,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

logging.info("Pipeline started")
logging.info(f"Dataset: {DATASET_NAME}")

# ------------------------------------------
# d) Load configuration from YAML
# ------------------------------------------

config_path = CONFIG_DIR / f"{CONFIG_NAME}.yaml"

if not config_path.exists():
    raise FileNotFoundError(
        f"\n[ERROR] Config file not found:\n{config_path}\n"
        f"Expected location: {CONFIG_DIR}\n"
        f"Make sure the YAML file exists and name is correct."
    )

with open(config_path, "r") as f:
    CONFIG = yaml.safe_load(f)

# ------------------------------------------
# e) Extract configuration sections
# ------------------------------------------

META = CONFIG.get("meta", {})
QC = CONFIG.get("qc", {})
ANALYSIS = CONFIG.get("analysis", {})
ANNOTATION = CONFIG.get("annotation", {})
VIS = CONFIG.get("visualization", {})

# ------------------------------------------
# f) Global settings
# ------------------------------------------

RANDOM_SEED = CONFIG.get("random_seed", 0)

# Full reproducibility
random.seed(RANDOM_SEED)

MULTI_RESOLUTIONS = CONFIG.get("multi_resolutions", [0.3, 0.5, 0.8, 1.0])
RANK_GENES_METHOD = CONFIG.get("rank_genes_method", "wilcoxon")
CLUSTER_KEY = CONFIG.get("cluster_key", "leiden")

# ------------------------------------------
# g) QC parameters
# ------------------------------------------

MIN_GENES_PER_CELL = QC.get("min_genes_per_cell")
MAX_GENES_PER_CELL = QC.get("max_genes_per_cell")
MAX_PCT_COUNTS_MT = QC.get("max_pct_mt")
MIN_CELLS_PER_GENE = QC.get("min_cells_per_gene")

# ------------------------------------------
# h) Analysis parameters
# ------------------------------------------

N_TOP_HVGS = ANALYSIS.get("n_top_hvgs")
N_PCS = ANALYSIS.get("n_pcs")
NEIGHBOR_K = ANALYSIS.get("neighbor_k")
DEFAULT_LEIDEN_RESOLUTION = ANALYSIS.get("leiden_resolution")

# ------------------------------------------
# i) Annotation & markers
# ------------------------------------------

REFERENCE_MARKERS = ANNOTATION.get("reference_markers", {})
MARKER_GENES_FOR_UMAP = VIS.get("umap_marker_genes", [])

# ------------------------------------------
# j) Summary printout (clean & readable)
# ------------------------------------------

print("====================================")
print("   Configuration Loaded Successfully")
print("====================================")
print(f"Dataset name:       {DATASET_NAME}")
print(f"Config file:        {config_path.name}")
print()

print("QC Parameters:")
print(f"  min_genes_per_cell: {MIN_GENES_PER_CELL}")
print(f"  max_genes_per_cell: {MAX_GENES_PER_CELL}")
print(f"  max_pct_mt:         {MAX_PCT_COUNTS_MT}")
print(f"  min_cells_per_gene: {MIN_CELLS_PER_GENE}")
print()

print("Analysis Parameters:")
print(f"  HVGs:       {N_TOP_HVGS}")
print(f"  PCs:        {N_PCS}")
print(f"  neighbors:  {NEIGHBOR_K}")
print(f"  leiden res: {DEFAULT_LEIDEN_RESOLUTION}")
print()

print(f"Annotation marker groups: {len(REFERENCE_MARKERS)}")
print(f"UMAP marker genes:        {len(MARKER_GENES_FOR_UMAP)}")

print("====================================")

logging.info("Configuration loaded successfully")

   Configuration Loaded Successfully
Dataset name:       260504-pbmc3k
Config file:        pbmc.yaml

QC Parameters:
  min_genes_per_cell: 200
  max_genes_per_cell: 2500
  max_pct_mt:         5.0
  min_cells_per_gene: 3

Analysis Parameters:
  HVGs:       2000
  PCs:        40
  neighbors:  15
  leiden res: 0.5

Annotation marker groups: 9
UMAP marker genes:        10


In [2]:
# ==========================================
# n1-2 — Configuration Validation (IMPROVED)
# ==========================================

def validate_config():

    print("Running configuration validation...\n")
    logging.info("Running configuration validation")

    # -------------------------
    # a) QC parameters
    # -------------------------

    required_qc = {
        "MIN_GENES_PER_CELL": MIN_GENES_PER_CELL,
        "MAX_GENES_PER_CELL": MAX_GENES_PER_CELL,
        "MAX_PCT_COUNTS_MT": MAX_PCT_COUNTS_MT,
        "MIN_CELLS_PER_GENE": MIN_CELLS_PER_GENE
    }

    for name, value in required_qc.items():
        if value is None:
            raise ValueError(f"[QC ERROR] Missing parameter: {name}")

    if MIN_GENES_PER_CELL is not None and MAX_GENES_PER_CELL is not None:
        if MIN_GENES_PER_CELL >= MAX_GENES_PER_CELL:
            raise ValueError(
                "[QC ERROR] MIN_GENES_PER_CELL must be smaller than MAX_GENES_PER_CELL"
            )

    if MAX_PCT_COUNTS_MT is not None:
        if not (0 <= MAX_PCT_COUNTS_MT <= 100):
            raise ValueError(
                "[QC ERROR] MAX_PCT_COUNTS_MT must be between 0 and 100"
            )

    if MIN_CELLS_PER_GENE <= 0:
        raise ValueError("[QC ERROR] MIN_CELLS_PER_GENE must be > 0")

    # -------------------------
    # b) Analysis parameters
    # -------------------------

    required_analysis = {
        "N_TOP_HVGS": N_TOP_HVGS,
        "N_PCS": N_PCS,
        "NEIGHBOR_K": NEIGHBOR_K,
        "DEFAULT_LEIDEN_RESOLUTION": DEFAULT_LEIDEN_RESOLUTION
    }

    for name, value in required_analysis.items():
        if value is None:
            raise ValueError(f"[ANALYSIS ERROR] Missing parameter: {name}")

    if N_TOP_HVGS <= 0:
        raise ValueError("[ANALYSIS ERROR] N_TOP_HVGS must be > 0")

    if N_PCS <= 0:
        raise ValueError("[ANALYSIS ERROR] N_PCS must be > 0")

    if NEIGHBOR_K <= 0:
        raise ValueError("[ANALYSIS ERROR] NEIGHBOR_K must be > 0")

    if DEFAULT_LEIDEN_RESOLUTION <= 0:
        raise ValueError("[ANALYSIS ERROR] Leiden resolution must be > 0")

    # -------------------------
    # c) Marker genes
    # -------------------------

    if not isinstance(REFERENCE_MARKERS, dict):
        raise TypeError("[ANNOTATION ERROR] REFERENCE_MARKERS must be a dictionary")

    for celltype, genes in REFERENCE_MARKERS.items():

        if not isinstance(genes, list):
            raise TypeError(
                f"[ANNOTATION ERROR] Markers for '{celltype}' must be a list"
            )

        if len(genes) == 0:
            raise ValueError(
                f"[ANNOTATION ERROR] Marker list for '{celltype}' is empty"
            )

        for g in genes:
            if not isinstance(g, str):
                raise TypeError(
                    f"[ANNOTATION ERROR] Gene '{g}' in '{celltype}' is not a string"
                )

    # -------------------------
    # d) UMAP markers
    # -------------------------

    if not isinstance(MARKER_GENES_FOR_UMAP, list):
        raise TypeError("[VIS ERROR] MARKER_GENES_FOR_UMAP must be a list")

    for g in MARKER_GENES_FOR_UMAP:
        if not isinstance(g, str):
            raise TypeError(f"[VIS ERROR] UMAP gene '{g}' is not a string")

    # -------------------------
    # e) Directory check
    # -------------------------

    if not DATA_DIR.exists():
        msg = f"[WARNING] Dataset directory does not exist yet: {DATA_DIR}"
        print(msg)
        logging.warning(msg)

    # -------------------------
    # f) Success message
    # -------------------------

    print("Configuration validation passed.\n")
    logging.info("Configuration validation passed")


# Run validation
validate_config()

Running configuration validation...

Configuration validation passed.



In [3]:
# -------------------------------------------------------------
# n1-2. Setup, imports, paths, and reproducibility (IMPROVED)
# -------------------------------------------------------------

import os
import sys
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets
import gseapy as gp

# ------------------------------------------------------------------
# a) Safety checks
# ------------------------------------------------------------------
if "BASE_DIR" not in globals():
    raise RuntimeError("BASE_DIR is not defined. Run configuration cell first.")

if "RESULTS_DIR" not in globals():
    raise RuntimeError("RESULTS_DIR is not defined.")

# ------------------------------------------------------------------
# b) Scanpy settings
# ------------------------------------------------------------------
sc.settings.verbosity = 3

# NOTE: n_jobs is deprecated in newer versions → avoid forcing it
# sc.settings.n_jobs = 1

sc.set_figure_params(
    dpi=300,
    frameon=False,
    facecolor="white",
    format="png"
)

# ------------------------------------------------------------------
# c) Display Paths & Summary
# ------------------------------------------------------------------
print("\n[INFO] Paths & Configuration")
print("-" * 50)
print("Current working directory:", Path.cwd())
print("Base directory:", BASE_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Figures directory:", FIG_DIR)

print(f"\nDataset: {DATASET_NAME}")
print(f"Cluster key: {CLUSTER_KEY}")
print(f"Leiden resolution: {DEFAULT_LEIDEN_RESOLUTION}")

# ------------------------------------------------------------------
# d) Reproducibility: seeds
# ------------------------------------------------------------------
np.random.seed(RANDOM_SEED)
sc.settings.seed = RANDOM_SEED

# ------------------------------------------------------------------
# e) Package versions
# ------------------------------------------------------------------
print("\n[INFO] Package versions")

CORE_PACKAGES = [
    "scanpy", "anndata", "pandas", "numpy",
    "matplotlib", "seaborn", "ipywidgets", "gseapy"
]

versions = {}

for pkg in CORE_PACKAGES:
    try:
        ver = importlib.metadata.version(pkg)
        versions[pkg] = ver
        print(f"  - {pkg}: {ver}")
    except importlib.metadata.PackageNotFoundError:
        versions[pkg] = "NOT INSTALLED"
        print(f"  - {pkg}: NOT INSTALLED")

print(f"  - python: {sys.version.split()[0]}")

# ------------------------------------------------------------------
# f) Save environment (IMPORTANT for GitHub reproducibility)
# ------------------------------------------------------------------
env_path = RESULTS_DIR / "0-environment.txt"

try:
    with open(env_path, "w") as f:
        f.write(f"python=={sys.version.split()[0]}\n")
        for pkg, ver in versions.items():
            f.write(f"{pkg}=={ver}\n")
    print(f"\n[OK] Environment saved → {env_path}")
except Exception as e:
    print(f"[WARNING] Could not save environment file: {e}")

# ------------------------------------------------------------------
# n1-2-2- Pretty printing (OPTIONAL, SAFE)
# ------------------------------------------------------------------

try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.table import Table
    from rich.text import Text

    console = Console()

    # -----------------------------
    # Project summary panel
    # -----------------------------
    header = Text()
    header.append("scRNA-seq Pipeline\n", style="bold cyan")
    header.append(f"Dataset: {DATASET_NAME}\n", style="bold white")
    header.append(f"Cluster key: {CLUSTER_KEY}\n", style="white")
    header.append(f"Leiden resolution: {DEFAULT_LEIDEN_RESOLUTION}", style="white")

    console.print(
        Panel(
            header,
            title="Project Summary",
            border_style="cyan",
            expand=False
        )
    )

    # -----------------------------
    # Paths table
    # -----------------------------
    paths_table = Table(title="Project Paths", show_header=True, header_style="bold magenta")

    paths_table.add_column("Name", style="bold")
    paths_table.add_column("Path", style="green")

    paths_table.add_row("Base directory", str(BASE_DIR))
    paths_table.add_row("Raw data", str(RAW_DATA_DIR))
    paths_table.add_row("Processed data", str(DATA_DIR))
    paths_table.add_row("Results", str(RESULTS_DIR))
    paths_table.add_row("Figures", str(FIG_DIR))

    console.print(paths_table)

    # -----------------------------
    # Parameters table
    # -----------------------------
    params_table = Table(title="Key Analysis Parameters", show_header=True, header_style="bold yellow")

    params_table.add_column("Parameter", style="bold")
    params_table.add_column("Value", style="white")

    params_table.add_row("Clustering method", "Leiden")
    params_table.add_row("Cluster key", CLUSTER_KEY)
    params_table.add_row("Default resolution", str(DEFAULT_LEIDEN_RESOLUTION))
    params_table.add_row("Random seed", str(RANDOM_SEED))

    console.print(params_table)

except ImportError:
    print("\n[INFO] 'rich' not installed → skipping pretty output")


[INFO] Paths & Configuration
--------------------------------------------------
Current working directory: K:\@scRNA-1.1-PBMC3K\notebooks
Base directory: K:\@scRNA-1.1-PBMC3K
Raw data directory: K:\@scRNA-1.1-PBMC3K\data\raw
Data directory: K:\@scRNA-1.1-PBMC3K\data\260504-pbmc3k
Results directory: K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k
Figures directory: K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k

Dataset: 260504-pbmc3k
Cluster key: leiden
Leiden resolution: 0.5

[INFO] Package versions
  - scanpy: 1.12
  - anndata: 0.12.10
  - pandas: 2.3.1
  - numpy: 2.2.5
  - matplotlib: 3.10.3
  - seaborn: 0.13.2
  - ipywidgets: 8.1.8
  - gseapy: 1.1.13
  - python: 3.13.2

[OK] Environment saved → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\0-environment.txt


╭─── Project Summary ────╮
│ scRNA-seq Pipeline     │
│ Dataset: 260504-pbmc3k │
│ Cluster key: leiden    │
│ Leiden resolution: 0.5 │
╰────────────────────────╯

                         Project Paths                         
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name           ┃ Path                                       ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Base directory │ K:\@scRNA-1.1-PBMC3K                       │
│ Raw data       │ K:\@scRNA-1.1-PBMC3K\data\raw              │
│ Processed data │ K:\@scRNA-1.1-PBMC3K\data\260504-pbmc3k    │
│ Results        │ K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k │
│ Figures        │ K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k │
└────────────────┴────────────────────────────────────────────┘

    Key Analysis Parameters    
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Parameter          ┃ Value  ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Clustering method  │ Leiden │
│ Cluster key        │ leiden │
│ Default resolution │ 0.5    │
│ Random seed        │ 0      │
└────────────────────┴────────┘

In [4]:
# ==========================================
# n1-3. Load raw 10x dataset (IMPROVED)
# ==========================================

import scanpy as sc
import pandas as pd
from pathlib import Path

print("\n[STEP] Loading 10x dataset...")

# ------------------------------------------------------------------
# a) Auto-detect 10x directory structure
# ------------------------------------------------------------------
def find_10x_path(base_path: Path):
    """
    Detect correct 10x folder (handles nested structures)
    """
    if (base_path / "matrix.mtx").exists():
        return base_path

    for sub in ["filtered_feature_bc_matrix", "raw_feature_bc_matrix"]:
        candidate = base_path / sub
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"No valid 10x matrix found in: {base_path}"
    )


try:
    tenx_path = find_10x_path(DATA_DIR)
    print(f"[INFO] Using 10x path: {tenx_path}")

    adata = sc.read_10x_mtx(
        tenx_path,
        var_names="gene_symbols",
        cache=True
    )

except Exception as e:
    raise RuntimeError(f"Failed to load 10x data: {e}")

# ------------------------------------------------------------------
# b) Basic cleanup
# ------------------------------------------------------------------
adata.var_names_make_unique()

# metadata
adata.uns["dataset"] = DATASET_NAME

# ------------------------------------------------------------------
# c) Summary
# ------------------------------------------------------------------
print("\n[INFO] AnnData loaded:")
print(f"  Cells: {adata.n_obs}")
print(f"  Genes: {adata.n_vars}")
print(adata)

# ------------------------------------------------------------------
# d) Pipeline metadata
# ------------------------------------------------------------------
adata.uns["pipeline"] = {
    "dataset": DATASET_NAME,
    "config": CONFIG_NAME,
    "date": pd.Timestamp.now().isoformat(),
    "random_seed": RANDOM_SEED,
    "scanpy_version": sc.__version__
}

# ------------------------------------------------------------------
# e) Save RAW checkpoint (compressed)
# ------------------------------------------------------------------
raw_h5ad_path = RESULTS_DIR / f"1-{DATASET_NAME}_raw.h5ad"

try:
    adata.write(raw_h5ad_path, compression="gzip")
    print(f"[OK] Raw AnnData saved → {raw_h5ad_path}")
except Exception as e:
    print(f"[WARNING] Could not save raw AnnData: {e}")

# ------------------------------------------------------------------
# f) Organism detection
# ------------------------------------------------------------------

def detect_organism_from_adata(adata, n_check=100):
    genes = list(adata.var_names[:n_check]) + list(adata.var_names[-n_check:])

    # Ensembl IDs
    if any(g.startswith("ENSG") for g in genes):
        return "human"
    if any(g.startswith("ENSMUSG") for g in genes):
        return "mouse"

    # gene symbol patterns
    human_like = sum(g.isupper() for g in genes)
    mouse_like = sum(g[:1].isupper() and g[1:].islower() for g in genes)

    if human_like > mouse_like:
        return "human"
    elif mouse_like > human_like:
        return "mouse"

    return "unknown"


def normalize_organism(org):
    org = org.lower().strip()

    mapping = {
        "homo sapiens": "human",
        "hsapiens": "human",
        "human": "human",
        "mus musculus": "mouse",
        "mmusculus": "mouse",
        "mouse": "mouse"
    }

    return mapping.get(org, org)


# --- detect ---
ORGANISM_DEFAULT = detect_organism_from_adata(adata)

if ORGANISM_DEFAULT == "unknown":
    print("[WARNING] Could not confidently detect organism.")
    print("[WARNING] Defaulting to 'human' → please verify manually.")
    ORGANISM_DEFAULT = "human"

# --- normalize ---
ORGANISM_DEFAULT = normalize_organism(ORGANISM_DEFAULT)

# --- store ---
adata.uns["organism"] = ORGANISM_DEFAULT

print(f"[INFO] Organism: {ORGANISM_DEFAULT}")


[STEP] Loading 10x dataset...
[INFO] Using 10x path: K:\@scRNA-1.1-PBMC3K\data\260504-pbmc3k
... reading from cache file cache\K-@scRNA-1.1-PBMC3K-data-260504-pbmc3k-matrix.h5ad

[INFO] AnnData loaded:
  Cells: 2700
  Genes: 32738
AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'
    uns: 'dataset'


C:\Users\espino\AppData\Local\Temp\ipykernel_5820\150854509.py:68: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  "scanpy_version": sc.__version__


[OK] Raw AnnData saved → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\1-260504-pbmc3k_raw.h5ad
[INFO] Organism: human


In [5]:
# ==========================================
# n1-4. Quality Control (QC) — IMPROVED
# ==========================================

print("\n[STEP] Quality Control (QC)")

# ------------------------------------------------------------------
# a) Define gene categories (organism-aware)
# ------------------------------------------------------------------
organism = adata.uns.get("organism", "human")

if organism == "human":
    mt_prefix = "MT-"
elif organism == "mouse":
    mt_prefix = "mt-"
else:
    mt_prefix = "MT-"  # fallback

# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith(mt_prefix)

# ribosomal genes (NEW → important)
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

# ------------------------------------------------------------------
# b) Compute QC metrics
# ------------------------------------------------------------------
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo"],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("[INFO] QC metrics calculated")

# ------------------------------------------------------------------
# c) QC summary
# ------------------------------------------------------------------
qc_columns = [
    "n_genes_by_counts",
    "total_counts",
    "pct_counts_mt",
    "pct_counts_ribo"
]

qc_summary = adata.obs[qc_columns].describe()

print("\n[QC Summary Statistics]")
print(qc_summary)

# ------------------------------------------------------------------
# d) Detect potential outliers (report only)
# ------------------------------------------------------------------
low_gene_cells = (adata.obs["n_genes_by_counts"] < MIN_GENES_PER_CELL).sum()
high_gene_cells = (adata.obs["n_genes_by_counts"] > MAX_GENES_PER_CELL).sum()
high_mt_cells = (adata.obs["pct_counts_mt"] > MAX_PCT_COUNTS_MT).sum()

print("\n[QC Flags]")
print(f"Cells with LOW genes (<{MIN_GENES_PER_CELL}): {low_gene_cells}")
print(f"Cells with HIGH genes (>{MAX_GENES_PER_CELL}): {high_gene_cells}")
print(f"Cells with HIGH MT% (>{MAX_PCT_COUNTS_MT}%): {high_mt_cells}")

# ------------------------------------------------------------------
# e) Visualization — Violin
# ------------------------------------------------------------------
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
    jitter=0.4,
    multi_panel=True,
    log=True,
    show=False
)

qc_violin_path = FIG_DIR / f"1-{DATASET_NAME}_QC_violin.png"
plt.savefig(qc_violin_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] QC violin → {qc_violin_path}")

# ------------------------------------------------------------------
# f) Scatter plots
# ------------------------------------------------------------------

# total_counts vs pct_counts_mt
sc.pl.scatter(
    adata,
    x="total_counts",
    y="pct_counts_mt",
    color="pct_counts_mt",
    show=False
)

qc_scatter_mt_path = FIG_DIR / f"2-{DATASET_NAME}_QC_pct_mt.png"
plt.savefig(qc_scatter_mt_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] QC scatter (MT) → {qc_scatter_mt_path}")

# total_counts vs n_genes
sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="n_genes_by_counts",
    show=False
)

qc_scatter_genes_path = FIG_DIR / f"3-{DATASET_NAME}_QC_genes.png"
plt.savefig(qc_scatter_genes_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] QC scatter (genes) → {qc_scatter_genes_path}")

# ------------------------------------------------------------------
# g) Save QC table
# ------------------------------------------------------------------
qc_table_path = RESULTS_DIR / f"2-{DATASET_NAME}_QC_metrics.csv"
adata.obs[qc_columns].to_csv(qc_table_path)

print(f"[OUTPUT] QC table saved → {qc_table_path}")


[STEP] Quality Control (QC)
[INFO] QC metrics calculated

[QC Summary Statistics]
       n_genes_by_counts  total_counts  pct_counts_mt  pct_counts_ribo
count        2700.000000   2700.000000    2700.000000      2700.000000
mean          846.994074   2366.900391       2.215132        34.949211
std           282.104964   1094.262085       1.165438        10.215296
min           212.000000    548.000000       0.000000         1.055966
25%           690.000000   1757.750000       1.536238        26.332963
50%           817.000000   2197.000000       2.029639        36.769516
75%           953.250000   2763.000000       2.640218        43.353328
max          3422.000000  15844.000000      22.569027        59.441711

[QC Flags]
Cells with LOW genes (<200): 0
Cells with HIGH genes (>2500): 5
Cells with HIGH MT% (>5.0%): 57
[FIG] QC violin → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\1-260504-pbmc3k_QC_violin.png
[FIG] QC scatter (MT) → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\2-260504-pbm

In [6]:
# ==========================================
# n1-5) Filtering + Normalization + HVG (FIXED FINAL)
# ==========================================

print("\n[STEP] Filtering & Normalization")

# ------------------------------------------------------------------
# a) Filter cells
# ------------------------------------------------------------------
n_before = adata.n_obs
sc.pp.filter_cells(adata, min_genes=MIN_GENES_PER_CELL)
print(f"[FILTER] min_genes: {n_before} → {adata.n_obs}")

if MAX_GENES_PER_CELL is not None:
    n_before = adata.n_obs
    adata = adata[adata.obs["n_genes_by_counts"] < MAX_GENES_PER_CELL, :].copy()
    print(f"[FILTER] max_genes: {n_before} → {adata.n_obs}")

if MAX_PCT_COUNTS_MT is not None:
    n_before = adata.n_obs
    adata = adata[adata.obs["pct_counts_mt"] < MAX_PCT_COUNTS_MT, :].copy()
    print(f"[FILTER] mt%: {n_before} → {adata.n_obs}")

# ------------------------------------------------------------------
# b) Filter genes
# ------------------------------------------------------------------
n_before = adata.n_vars
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
print(f"[FILTER] genes: {n_before} → {adata.n_vars}")

print(adata)

# ------------------------------------------------------------------
# c) Save QC checkpoint
# ------------------------------------------------------------------
qc_filtered_path = RESULTS_DIR / f"2-{DATASET_NAME}_QC_filtered.h5ad"
adata.write(qc_filtered_path, compression="gzip")
print(f"[OK] QC-filtered saved → {qc_filtered_path}")

# ------------------------------------------------------------------
# d) Save raw counts BEFORE normalization  ✅ (CRITICAL)
# ------------------------------------------------------------------
adata.layers["counts"] = adata.X.copy()

# ------------------------------------------------------------------
# e) Normalize + log
# ------------------------------------------------------------------
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# store normalized (Scanpy convention)
adata.raw = adata

# save
norm_log_path = RESULTS_DIR / f"3-{DATASET_NAME}_norm_log.h5ad"
adata.write(norm_log_path, compression="gzip")
print(f"[OK] Normalized saved → {norm_log_path}")

# ------------------------------------------------------------------
# f) HVGs (FIXED)
# ------------------------------------------------------------------
print("[INFO] Computing HVGs on raw counts...")

sc.pp.highly_variable_genes(
    adata,
    flavor="seurat_v3",
    n_top_genes=N_TOP_HVGS,
    layer="counts"   # 🔥🔥🔥 FIX HERE
)

print(f"[INFO] HVGs: {adata.var['highly_variable'].sum()}")

# plot
sc.pl.highly_variable_genes(adata, show=False)
hvg_fig_path = FIG_DIR / f"4-{DATASET_NAME}_HVG.png"
plt.savefig(hvg_fig_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] HVG plot → {hvg_fig_path}")

# ------------------------------------------------------------------
# g) Subset to HVGs
# ------------------------------------------------------------------
adata = adata[:, adata.var["highly_variable"]].copy()

print("[INFO] After HVG subset:")
print(adata)


[STEP] Filtering & Normalization
[FILTER] min_genes: 2700 → 2700
[FILTER] max_genes: 2700 → 2695
[FILTER] mt%: 2695 → 2638
filtered out 19082 genes that are detected in less than 3 cells
[FILTER] genes: 32738 → 13656
AnnData object with n_obs × n_vars = 2638 × 13656
    obs: 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'n_genes'
    var: 'gene_ids', 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'dataset', 'pipeline', 'organism'
[OK] QC-filtered saved → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\2-260504-pbmc3k_QC_filtered.h5ad
normalizing counts per cell
    finished (0:00:19)
[OK] Normalized saved → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\3-260504-pbmc3k_norm_log.h5ad
[INFO] Computing HVGs on raw counts...
extracting highly variable genes
--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
 

In [7]:
# =========================================================
# n1-6) Scaling, PCA, Neighbors (FIXED + ROBUST)
# =========================================================

print("\n[STEP] Scaling + PCA + Neighbors")

# ---------------------------------------------------------
# 1) HVG FIX (IMPORTANT — run on raw counts)
# ---------------------------------------------------------
print("[INFO] Recomputing HVGs on raw counts...")

sc.pp.highly_variable_genes(
    adata,
    flavor="seurat_v3",
    n_top_genes=N_TOP_HVGS,
    layer="counts"   # 🔥 FIX: use raw counts
)

print(f"[INFO] HVGs: {adata.var['highly_variable'].sum()}")

# Plot HVGs
sc.pl.highly_variable_genes(adata, show=False)
hvg_fig_path = FIG_DIR / f"4-{DATASET_NAME}_HVG.png"
plt.savefig(hvg_fig_path, bbox_inches="tight")
plt.close()
print(f"[FIG] HVG plot → {hvg_fig_path}")

# Subset to HVGs
adata = adata[:, adata.var["highly_variable"]].copy()
print(f"[INFO] After HVG subset: {adata.shape}")


# ---------------------------------------------------------
# 2) Scaling
# ---------------------------------------------------------
print("[INFO] Scaling data...")

# Optional but recommended (can improve biological signal)
# Uncomment if needed:
# sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"])

#sc.pp.scale(adata, max_value=10)
sc.pp.scale(adata, max_value=10, zero_center=True)

# ---------------------------------------------------------
# 3) PCA
# ---------------------------------------------------------
print("[INFO] Running PCA...")

sc.tl.pca(
    adata,
    svd_solver="arpack",
    n_comps=N_PCS
)

# PCA variance plot
sc.pl.pca_variance_ratio(
    adata,
    log=True,
    n_pcs=N_PCS,
    show=False
)

pca_var_fig_path = FIG_DIR / f"5-{DATASET_NAME}_PCA_variance.png"
plt.savefig(pca_var_fig_path, bbox_inches="tight")
plt.close()

print(f"[FIG] PCA variance → {pca_var_fig_path}")


# ---------------------------------------------------------
# 4) Neighbors graph
# ---------------------------------------------------------
print("[INFO] Computing neighborhood graph...")

sc.pp.neighbors(
    adata,
    n_neighbors=NEIGHBOR_K,
    n_pcs=N_PCS
)

# Save checkpoint
neighbors_path = RESULTS_DIR / f"4-{DATASET_NAME}_neighbors.h5ad"
adata.write(neighbors_path)

print(f"[OK] Neighbors saved → {neighbors_path}")
print("[STEP COMPLETED]")


[STEP] Scaling + PCA + Neighbors
[INFO] Recomputing HVGs on raw counts...
extracting highly variable genes
--> added
    'highly_variable', boolean vector (adata.var)
    'highly_variable_rank', float vector (adata.var)
    'means', float vector (adata.var)
    'variances', float vector (adata.var)
    'variances_norm', float vector (adata.var)
[INFO] HVGs: 2000
[FIG] HVG plot → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\4-260504-pbmc3k_HVG.png
[INFO] After HVG subset: (2638, 2000)
[INFO] Scaling data...


C:\Users\espino\AppData\Local\Programs\Python\Python313\Lib\functools.py:934: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


[INFO] Running PCA...
computing PCA
    with n_comps=40
    finished (0:00:09)
[FIG] PCA variance → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\5-260504-pbmc3k_PCA_variance.png
[INFO] Computing neighborhood graph...
computing neighbors
    using 'X_pca' with n_pcs = 40
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:01:08)
[OK] Neighbors saved → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\4-260504-pbmc3k_neighbors.h5ad
[STEP COMPLETED]


In [8]:
# =========================================================
# n1-9) UMAP + Clustering (IMPROVED)
# =========================================================

print("\n[STEP] UMAP + Leiden clustering")

# ---------------------------------------------------------
# 1) UMAP
# ---------------------------------------------------------
print("[INFO] Computing UMAP embedding...")

sc.tl.umap(adata, random_state=RANDOM_SEED)

sc.pl.umap(
    adata,
    color=["n_genes_by_counts", "pct_counts_mt"],
    frameon=False,
    show=False
)

umap_fig_path = FIG_DIR / f"6-{DATASET_NAME}_UMAP_QC.png"
plt.savefig(umap_fig_path, bbox_inches="tight")
plt.close()

print(f"[FIG] UMAP QC → {umap_fig_path}")


# ---------------------------------------------------------
# 2) Marker gene UMAP (cleaner)
# ---------------------------------------------------------
if len(MARKER_GENES_FOR_UMAP) > 0:

    genes_in_data = [g for g in MARKER_GENES_FOR_UMAP if g in adata.var_names]

    if genes_in_data:
        print(f"[INFO] Plotting {len(genes_in_data)} marker genes")

        sc.pl.umap(
            adata,
            color=genes_in_data,
            frameon=False,
            show=False
        )

        marker_path = FIG_DIR / f"8-{DATASET_NAME}_UMAP_markers.png"
        plt.savefig(marker_path, bbox_inches="tight")
        plt.close()

        print(f"[FIG] Marker UMAP → {marker_path}")
    else:
        print("[WARN] No marker genes found in dataset")


# ---------------------------------------------------------
# 3) Leiden clustering (main)
# ---------------------------------------------------------
print("[INFO] Running Leiden clustering...")

sc.tl.leiden(
    adata,
    resolution=DEFAULT_LEIDEN_RESOLUTION,
    key_added=CLUSTER_KEY,
    flavor="igraph",
    n_iterations=-1,
    directed=False
)

print(f"[INFO] Clusters stored in: adata.obs['{CLUSTER_KEY}']")

sc.pl.umap(
    adata,
    color=[CLUSTER_KEY],
    legend_loc="on data",
    frameon=False,
    show=False
)

leiden_path = FIG_DIR / f"7-{DATASET_NAME}_UMAP_Leiden.png"
plt.savefig(leiden_path, bbox_inches="tight")
plt.close()

print(f"[FIG] Leiden UMAP → {leiden_path}")


# ---------------------------------------------------------
# 4) Multi-resolution clustering
# ---------------------------------------------------------
print("[INFO] Running multi-resolution Leiden...")

for r in MULTI_RESOLUTIONS:
    key = f"leiden_{r}"

    sc.tl.leiden(
        adata,
        resolution=r,
        key_added=key,
        flavor="igraph",
        n_iterations=-1,
        directed=False
    )

    print(f"[OK] resolution={r} → stored in {key}")


# Plot multi-resolution
sc.pl.umap(
    adata,
    color=[f"leiden_{r}" for r in MULTI_RESOLUTIONS],
    frameon=False,
    show=False
)

multi_path = FIG_DIR / f"8-{DATASET_NAME}_UMAP_multi_resolution.png"
plt.savefig(multi_path, bbox_inches="tight")
plt.close()

print(f"[FIG] Multi-resolution UMAP → {multi_path}")


# ---------------------------------------------------------
# 5) Dendrogram (cluster relationships)
# ---------------------------------------------------------
sc.tl.dendrogram(adata, groupby=CLUSTER_KEY)

print("[STEP COMPLETED]")


[STEP] UMAP + Leiden clustering
[INFO] Computing UMAP embedding...
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:20)
[FIG] UMAP QC → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\6-260504-pbmc3k_UMAP_QC.png
[INFO] Plotting 7 marker genes
[FIG] Marker UMAP → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\8-260504-pbmc3k_UMAP_markers.png
[INFO] Running Leiden clustering...
running Leiden clustering
    finished: found 6 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:03)
[INFO] Clusters stored in: adata.obs['leiden']
[FIG] Leiden UMAP → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\7-260504-pbmc3k_UMAP_Leiden.png
[INFO] Running multi-resolution Leiden...
running Leiden clustering
    finished: found 4 clusters and added
    'leiden_0.3', the cluster labels (adata.obs, categorical) (0:00:00)
[OK] resolution=0.3 → stored in leiden_0.3
running Leiden clustering
    finished: found 6 clus

In [9]:
# ==========================================
# n1-10) Marker gene analysis (IMPROVED + PUBLICATION READY)
# ==========================================

print("\n[STEP] Marker Gene Analysis")

# ------------------------------------------------------------------
# a) Rank genes per cluster
# ------------------------------------------------------------------
sc.tl.rank_genes_groups(
    adata,
    groupby=CLUSTER_KEY,
    method=RANK_GENES_METHOD,
    pts=True
)

print(f"[INFO] Marker analysis done ({RANK_GENES_METHOD}) on '{CLUSTER_KEY}'")

# ------------------------------------------------------------------
# b) Convert results to DataFrame
# ------------------------------------------------------------------
markers_df = sc.get.rank_genes_groups_df(
    adata,
    group=None
)

# ------------------------------------------------------------------
# c) Clean & filter markers (IMPORTANT)
# ------------------------------------------------------------------
# Remove NaNs
markers_df = markers_df.dropna()

# Add significance filters
markers_df = markers_df[
    (markers_df["pvals_adj"] < 0.05) &
    (markers_df["logfoldchanges"].abs() > 0.25)
]

# Sort by cluster + score
markers_df = markers_df.sort_values(
    ["group", "scores"],
    ascending=[True, False]
)

print(f"[INFO] Filtered markers: {markers_df.shape[0]} rows")

# ------------------------------------------------------------------
# d) Save full table
# ------------------------------------------------------------------
markers_csv_path = RESULTS_DIR / f"5-{DATASET_NAME}_marker_genes_filtered.csv"
markers_df.to_csv(markers_csv_path, index=False)

print(f"[OK] Marker table saved → {markers_csv_path}")

# ------------------------------------------------------------------
# e) Top markers per cluster (for plotting)
# ------------------------------------------------------------------
top_markers = (
    markers_df
    .groupby("group")
    .head(10)
)

# ------------------------------------------------------------------
# f) Dotplot (publication-friendly)
# ------------------------------------------------------------------
sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=5,
    groupby=CLUSTER_KEY,
    show=False
)

dotplot_path = FIG_DIR / f"9-{DATASET_NAME}_marker_dotplot.png"
plt.savefig(dotplot_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] Dotplot → {dotplot_path}")

# ------------------------------------------------------------------
# g) Heatmap (strong figure for papers)
# ------------------------------------------------------------------
sc.pl.rank_genes_groups_heatmap(
    adata,
    n_genes=5,
    groupby=CLUSTER_KEY,
    show=False
)

heatmap_path = FIG_DIR / f"10-{DATASET_NAME}_marker_heatmap.png"
plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] Heatmap → {heatmap_path}")

# ------------------------------------------------------------------
# h) Classical bar plot (original)
# ------------------------------------------------------------------
sc.pl.rank_genes_groups(
    adata,
    n_genes=20,
    sharey=False,
    show=False
)

barplot_path = FIG_DIR / f"11-{DATASET_NAME}_Top_marker_genes.png"
plt.savefig(barplot_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] Barplot → {barplot_path}")

# ------------------------------------------------------------------
# i) Save processed AnnData (checkpoint)
# ------------------------------------------------------------------
processed_h5ad_path = RESULTS_DIR / f"6-{DATASET_NAME}_adata_processed.h5ad"
adata.write(processed_h5ad_path, compression="gzip")

print(f"[OK] Processed AnnData saved → {processed_h5ad_path}")

print("\n[STEP COMPLETED]")


[STEP] Marker Gene Analysis
ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:06)
[INFO] Marker analysis done (wilcoxon) on 'leiden'
[INFO] Filtered markers: 4353 rows
[OK] Marker table saved → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\5-260504-pbmc3k_marker_genes_filtered.csv
[FIG] Dotplot → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\9-260504-pbmc3k_marker_dotplot.png
[FIG] Heatmap → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\10-260504-pbmc3k_marker_heatmap.png
[FIG] Barplot → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\11-260504-pbmc3k_Top_marker_genes.png
[OK] Processed AnnData saved → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\6-260504-pbmc3k_adata_processe

In [10]:
## END of notebook 1- Analysis

In [11]:
## ---------------------------------------

In [12]:
## notebook 2 - Annotation 

In [13]:
# ==========================================
# n2-11) Cell-type annotation (FINAL CLEAN VERSION)
# ==========================================

print("\n[STEP] Cell-type annotation")

import numpy as np

# ------------------------------------------------------------------
# a) Use raw expression (CORRECT)
# ------------------------------------------------------------------
use_raw_flag = adata.raw is not None

print(f"[INFO] Using raw: {use_raw_flag}")

# ------------------------------------------------------------------
# b) Score genes
# ------------------------------------------------------------------
celltypes = []

for ct, genes in REFERENCE_MARKERS.items():

    present = [g for g in genes if g in adata.var_names]

    if len(present) == 0:
        print(f"[WARN] No markers found for {ct}")
        continue

    print(f"[INFO] {ct}: {len(present)} markers")

    score_key = f"score_{ct}"

    sc.tl.score_genes(
        adata,
        gene_list=present,
        score_name=score_key,
        use_raw=use_raw_flag
    )

    celltypes.append(ct)

# ------------------------------------------------------------------
# c) Assign cell type (NO normalization!)
# ------------------------------------------------------------------
score_cols = [f"score_{ct}" for ct in celltypes]
score_matrix = adata.obs[score_cols].values

best_idx = np.argmax(score_matrix, axis=1)
best_labels = np.array(celltypes)[best_idx]

# Optional simple threshold (SAFE)
SCORE_THRESHOLD = 0  # یا None

if SCORE_THRESHOLD is not None:
    best_scores = score_matrix[np.arange(score_matrix.shape[0]), best_idx]
    best_labels = [
        ct if s > SCORE_THRESHOLD else "Unknown"
        for ct, s in zip(best_labels, best_scores)
    ]

adata.obs["cell_type"] = best_labels

# ------------------------------------------------------------------
# d) Cluster-level annotation
# ------------------------------------------------------------------
cluster_majority = (
    adata.obs.groupby(CLUSTER_KEY)["cell_type"]
    .agg(lambda x: x.value_counts().idxmax())
)

adata.obs["cell_type_cluster"] = adata.obs[CLUSTER_KEY].map(cluster_majority)

# ------------------------------------------------------------------
# e) Summary
# ------------------------------------------------------------------
print("\n[Cell type counts]")
print(adata.obs["cell_type"].value_counts())

# ------------------------------------------------------------------
# f) UMAP
# ------------------------------------------------------------------
sc.pl.umap(
    adata,
    color=["cell_type"],
    legend_loc="right margin",
    legend_fontsize=8,
    frameon=False,
    show=False
)

path = FIG_DIR / f"12-{DATASET_NAME}_UMAP_cell_types.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.close()

print(f"[FIG] UMAP → {path}")

# ------------------------------------------------------------------
# g) Save
# ------------------------------------------------------------------
path_h5ad = RESULTS_DIR / f"7-{DATASET_NAME}_adata_annotated.h5ad"
adata.write(path_h5ad, compression="gzip")

print(f"[OK] Saved → {path_h5ad}")

print("\n[STEP COMPLETED]")


[STEP] Cell-type annotation
[INFO] Using raw: True
[INFO] CD4_T: 1 markers
computing score 'score_CD4_T'
    finished: added
    'score_CD4_T', score of gene set (adata.obs).
    50 total control genes are used. (0:00:00)
[INFO] CD8_T: 1 markers
computing score 'score_CD8_T'
    finished: added
    'score_CD8_T', score of gene set (adata.obs).
    49 total control genes are used. (0:00:00)
[INFO] NK_cells: 3 markers
computing score 'score_NK_cells'
    finished: added
    'score_NK_cells', score of gene set (adata.obs).
    99 total control genes are used. (0:00:00)
[INFO] Naive_B: 2 markers
computing score 'score_Naive_B'
    finished: added
    'score_Naive_B', score of gene set (adata.obs).
    50 total control genes are used. (0:00:00)
[INFO] Memory_B: 2 markers
computing score 'score_Memory_B'
    finished: added
    'score_Memory_B', score of gene set (adata.obs).
    50 total control genes are used. (0:00:00)
[INFO] CD14_Monocytes: 3 markers
computing score 'score_CD14_Monocyte

C:\Users\espino\AppData\Local\Temp\ipykernel_5820\1273383737.py:67: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(CLUSTER_KEY)["cell_type"]
... storing 'cell_type' as categorical



[Cell type counts]
cell_type
CD4_T               555
CD14_Monocytes      478
Naive_B             377
Memory_B            339
Unknown             246
CD8_T               208
FCGR3A_Monocytes    191
NK_cells            169
Dendritic_cells      43
Platelets            32
Name: count, dtype: int64
[FIG] UMAP → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\12-260504-pbmc3k_UMAP_cell_types.png
[OK] Saved → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\7-260504-pbmc3k_adata_annotated.h5ad

[STEP COMPLETED]


In [14]:
# ============================================================
# n2-13) Supplementary Annotation Quality & Marker Analysis (IMPROVED)
# ============================================================

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

print("\n[STEP] Supplementary Analysis")

# --------------------------------------------------
# a) Annotation confidence (robust)
# --------------------------------------------------

score_cols = [c for c in adata.obs.columns if c.startswith("score_")]

if len(score_cols) >= 2:
    scores = adata.obs[score_cols].to_numpy()

    # z-score per cell (robust scaling)
    scores = (scores - scores.mean(axis=1, keepdims=True)) / \
             (scores.std(axis=1, keepdims=True) + 1e-9)

    sorted_scores = np.sort(scores, axis=1)

    # difference between top1 and top2
    confidence = sorted_scores[:, -1] - sorted_scores[:, -2]

    adata.obs["annotation_confidence"] = confidence

    print("[OK] annotation_confidence computed (z-score based)")
else:
    adata.obs["annotation_confidence"] = np.nan
    print("[WARN] Not enough score columns")

# --------------------------------------------------
# b) Cluster marker genes (safe)
# --------------------------------------------------

markers_csv = RESULTS_DIR / f"8-{DATASET_NAME}_cluster_markers.csv"

if CLUSTER_KEY in adata.obs:

    if "rank_genes_groups" not in adata.uns:
        print("[INFO] Computing cluster markers...")
        sc.tl.rank_genes_groups(
            adata,
            groupby=CLUSTER_KEY,
            method=RANK_GENES_METHOD,
            n_genes=50
        )
    else:
        print("[INFO] Using existing rank_genes_groups")

    markers_df = sc.get.rank_genes_groups_df(adata, group=None)
    markers_df.to_csv(markers_csv, index=False)

    print(f"[OK] Marker table → {markers_csv}")

# --------------------------------------------------
# c) Cluster vs Cell-Type (table + heatmap)
# --------------------------------------------------

if CLUSTER_KEY in adata.obs and "cell_type" in adata.obs:

    ctab = pd.crosstab(
        adata.obs[CLUSTER_KEY],
        adata.obs["cell_type"]
    )

    ctab_path = RESULTS_DIR / f"9-{DATASET_NAME}_cluster_celltype_table.csv"
    ctab.to_csv(ctab_path)

    print(f"[OK] Crosstab → {ctab_path}")

    # 🔥 NEW: Heatmap (VERY IMPORTANT for reports)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        ctab,
        cmap="viridis",
        annot=True,
        fmt="d"
    )

    heatmap_path = FIG_DIR / f"12-{DATASET_NAME}_cluster_celltype_heatmap.png"
    plt.title("Cluster vs Cell Type")
    plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"[FIG] Heatmap → {heatmap_path}")

    # consistency summary
    summary = []

    for cluster, row in ctab.iterrows():
        total = row.sum()
        if total == 0:
            continue

        top_ct = row.idxmax()
        frac = row.max() / total

        summary.append({
            "cluster": cluster,
            "dominant_cell_type": top_ct,
            "fraction": frac
        })

    summary_df = pd.DataFrame(summary)

    summary_path = RESULTS_DIR / f"10-{DATASET_NAME}_cluster_consistency.csv"
    summary_df.to_csv(summary_path, index=False)

    print(f"[OK] Consistency → {summary_path}")

# --------------------------------------------------
# d) UMAP confidence (clean visualization)
# --------------------------------------------------

if "annotation_confidence" in adata.obs:

    conf_path = FIG_DIR / f"11-{DATASET_NAME}_UMAP_confidence.png"

    sc.pl.umap(
        adata,
        color="annotation_confidence",
        cmap="viridis",
        frameon=False,
        show=False
    )

    plt.savefig(conf_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"[FIG] UMAP confidence → {conf_path}")

print("[STEP COMPLETED]\n")


[STEP] Supplementary Analysis
[OK] annotation_confidence computed (z-score based)
[INFO] Using existing rank_genes_groups
[OK] Marker table → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\8-260504-pbmc3k_cluster_markers.csv
[OK] Crosstab → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\9-260504-pbmc3k_cluster_celltype_table.csv
[FIG] Heatmap → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\12-260504-pbmc3k_cluster_celltype_heatmap.png
[OK] Consistency → K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\10-260504-pbmc3k_cluster_consistency.csv
[FIG] UMAP confidence → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\11-260504-pbmc3k_UMAP_confidence.png
[STEP COMPLETED]



In [15]:
# =========================================================
# n2-14) Automated Analysis Report (PDF)
# =========================================================

from pathlib import Path
from datetime import datetime

import pandas as pd

# -----------------------------
# Robust fallbacks for globals
# -----------------------------
CONFIG = globals().get("CONFIG", {})
META = globals().get("META", {})
QC = globals().get("QC", {})
ANALYSIS = globals().get("ANALYSIS", {})
ANNOTATION = globals().get("ANNOTATION", {})
VIS = globals().get("VIS", {})

DATASET_NAME = globals().get("DATASET_NAME", META.get("project_name", "dataset"))
RESULTS_DIR = globals().get("RESULTS_DIR", Path("results") / DATASET_NAME)
FIG_DIR = globals().get("FIG_DIR", Path("figures") / DATASET_NAME)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# ReportLab imports
# -----------------------------
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image,
    Table,
    TableStyle,
)
from reportlab.lib.styles import getSampleStyleSheet

print("Building automated PDF report (Step 14)...")

report_path = RESULTS_DIR / f"11-{DATASET_NAME}_scRNAseq_analysis_report_pro.pdf"
doc = SimpleDocTemplate(
    str(report_path),
    pagesize=A4,
    leftMargin=2 * cm,
    rightMargin=2 * cm,
    topMargin=2 * cm,
    bottomMargin=2 * cm,
)

styles = getSampleStyleSheet()
style_title = styles["Title"]
style_heading = styles["Heading2"]
style_body = styles["BodyText"]

story = []

# =========================================================
# 14.1 Title page
# =========================================================
title_text = f"scRNA-seq Analysis Report<br/>{DATASET_NAME}"
story.append(Paragraph(title_text, style_title))
story.append(Spacer(1, 0.7 * cm))

meta_lines = []
meta_lines.append(f"Project name: {META.get('project_name', DATASET_NAME)}")
meta_lines.append(f"Tissue / dataset type: {META.get('tissue_type', CONFIG.get('config_name', 'N/A'))}")
meta_lines.append(f"Species: {META.get('species', 'N/A')}")
meta_lines.append(f"Date: {datetime.now().strftime('%Y-%m-%d')}")
meta_lines.append(f"Results directory: {RESULTS_DIR}")
meta_lines.append(f"Figures directory: {FIG_DIR}")
meta_text = "<br/>".join(meta_lines)
story.append(Paragraph(meta_text, style_body))
story.append(Spacer(1, 1.0 * cm))

# =========================================================
# 14.2 Pipeline overview
# =========================================================
story.append(Paragraph("1. Pipeline Overview", style_heading))
overview = (
    "This report summarizes a standardized single-cell RNA-seq analysis pipeline, including: "
    "quality control, normalization and log-transformation, dimensionality reduction (PCA and UMAP), "
    "graph-based clustering (Leiden), marker-gene analysis, automatic cell-type annotation based on "
    "reference markers, and supplementary evaluation of annotation confidence and cluster consistency."
)
story.append(Paragraph(overview, style_body))
story.append(Spacer(1, 0.5 * cm))

# =========================================================
# 14.3 QC parameters + QC figures
# =========================================================
story.append(Paragraph("2. Quality Control (QC)", style_heading))

qc_lines = []
qc_lines.append("<b>QC parameters (from YAML)</b>")
for key in ["min_genes_per_cell", "max_genes_per_cell", "max_pct_mt", "min_cells_per_gene"]:
    if key in QC:
        qc_lines.append(f"{key}: {QC[key]}")
qc_text = "<br/>".join(qc_lines) if qc_lines else "QC parameters not available."
story.append(Paragraph(qc_text, style_body))
story.append(Spacer(1, 0.3 * cm))

# Try to embed QC figures if they exist
qc_figs = [
    FIG_DIR / f"1-{DATASET_NAME}_QC_violin.png",
    FIG_DIR / f"2-{DATASET_NAME}_QC_pct_counts_mt.png",
    FIG_DIR / f"3-{DATASET_NAME}_QC_n_genes_by_counts.png",
]
for fig_path in qc_figs:
    if fig_path.is_file():
        story.append(Image(str(fig_path), width=14 * cm, height=8 * cm))
        story.append(Spacer(1, 0.3 * cm))

# =========================================================
# 14.4 UMAPs & clustering
# =========================================================
story.append(Paragraph("3. UMAP Embeddings & Clustering", style_heading))

umap_text = (
    "UMAP embeddings provide a 2D representation of the high-dimensional transcriptomic space. "
    "Cells are colored by QC status, clusters, or cell types to illustrate global structure and "
    "major cell populations."
)
story.append(Paragraph(umap_text, style_body))
story.append(Spacer(1, 0.3 * cm))

umap_figs = [
    FIG_DIR / f"6-{DATASET_NAME}_umap_qc.png",
    FIG_DIR / f"7-{DATASET_NAME}_Leiden_clustering_UMAP.png",
    FIG_DIR / f"8-{DATASET_NAME}_UMAP_marker_genes.png",
    FIG_DIR / f"11-{DATASET_NAME}_UMAP_annotation_confidence.png",
]
for fig_path in umap_figs:
    if fig_path.is_file():
        story.append(Image(str(fig_path), width=14 * cm, height=8 * cm))
        story.append(Spacer(1, 0.3 * cm))

# =========================================================
# 14.5 Cell-type annotation & marker genes
# =========================================================
story.append(Paragraph("4. Cell-Type Annotation & Marker Genes", style_heading))

ref_markers = ANNOTATION.get("reference_markers", {})
if ref_markers:
    n_ct = len(ref_markers)
    story.append(
        Paragraph(
            f"Automatic cell-type annotation was performed using {n_ct} reference marker sets "
            "defined in the YAML configuration. Each cell was assigned the label whose marker "
            "gene score was highest.",
            style_body,
        )
    )
else:
    story.append(
        Paragraph(
            "No reference markers were defined in the YAML configuration; annotation may be performed manually.",
            style_body,
        )
    )
story.append(Spacer(1, 0.3 * cm))

# Top markers table from cluster marker CSV (Step 13)
markers_csv_path = RESULTS_DIR / f"8-{DATASET_NAME}_cluster_markers.csv"
if markers_csv_path.is_file():
    try:
        markers_df = pd.read_csv(markers_csv_path)
        # take top 3 markers per cluster for a compact table
        top_markers = markers_df.groupby("group").head(3)

        table_data = [["cluster", "gene", "logFC", "pval_adj"]]
        for _, row in top_markers.iterrows():
            table_data.append(
                [
                    str(row.get("group", "")),
                    str(row.get("names", "")),
                    f"{row.get('logfoldchanges', 0):.2f}",
                    f"{row.get('pvals_adj', 0):.1e}",
                ]
            )

        tbl = Table(table_data, hAlign="LEFT")
        tbl.setStyle(
            TableStyle(
                [
                    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
                    ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
                    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ]
            )
        )
        story.append(tbl)
        story.append(Spacer(1, 0.3 * cm))
    except Exception as e:
        story.append(Paragraph(f"Could not load marker table: {e}", style_body))

# =========================================================
# 14.6 Cluster vs cell-type consistency
# =========================================================
summary_path = RESULTS_DIR / f"10-{DATASET_NAME}_cluster_annotation_consistency.csv"
if summary_path.is_file():
    try:
        summary_df = pd.read_csv(summary_path)
        story.append(Paragraph("5. Cluster vs Cell-Type Consistency", style_heading))

        table_data = [["cluster", "dominant cell type", "n_cells", "fraction_dominant"]]
        for _, row in summary_df.iterrows():
            table_data.append(
                [
                    str(row["cluster"]),
                    str(row["dominant_cell_type"]),
                    int(row["n_cells"]),
                    f"{row['fraction_dominant']:.2f}",
                ]
            )
        tbl = Table(table_data, hAlign="LEFT")
        tbl.setStyle(
            TableStyle(
                [
                    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
                    ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
                    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ]
            )
        )
        story.append(tbl)
        story.append(Spacer(1, 0.3 * cm))
    except Exception as e:
        story.append(Paragraph(f"Could not load cluster-consistency table: {e}", style_body))

# =========================================================
# 14.7 Appendix — key parameters from YAML
# =========================================================
story.append(Paragraph("6. Key Analysis Parameters (from YAML)", style_heading))

param_lines = []

param_lines.append("<b>QC</b>")
for k, v in QC.items():
    param_lines.append(f"{k}: {v}")

param_lines.append("<br/><b>Analysis</b>")
for k, v in ANALYSIS.items():
    param_lines.append(f"{k}: {v}")

param_lines.append("<br/><b>Annotation</b>")
param_lines.append(f"reference marker groups: {len(ref_markers)}")

param_text = "<br/>".join(param_lines) if param_lines else "No parameters available."
story.append(Paragraph(param_text, style_body))

story.append(Paragraph("Total cells: {}".format(adata.n_obs), style_body))
story.append(Paragraph("Total genes: {}".format(adata.n_vars), style_body))

# =========================================================
# Build document
# =========================================================
doc.build(story)
print(f"PDF report saved to: {report_path}\n")


Building automated PDF report (Step 14)...
PDF report saved to: K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\11-260504-pbmc3k_scRNAseq_analysis_report_pro.pdf



In [16]:
## END of notebook 2- Annotation

In [17]:
# --------------------------------------------

In [18]:
## notebook 3 - GO Enrichment Analysis (BP, MF, CC) 

In [19]:
# =========================================================
# n3-00) Resource Manager Setup
# =========================================================

from pathlib import Path
from datetime import datetime

RESOURCE_DIR = BASE_DIR / "resources"
GO_DIR = RESOURCE_DIR / "go"
KEGG_DIR = RESOURCE_DIR / "kegg"
META_DIR = RESOURCE_DIR / "metadata"
LOG_DIR = RESOURCE_DIR / "logs"

for d in [GO_DIR, KEGG_DIR, META_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("[OK] Resource directories initialized")

[OK] Resource directories initialized


In [20]:
'''
# =========================================================
# n3-01) Build Gene Set Registry (OFFLINE MODE)
# =========================================================

import pandas as pd
from datetime import datetime

registry = []

def add_entry(name, path, category, source, organism="human"):
    registry.append({
        "name": name,
        "path": str(path),
        "category": category,
        "source": source,
        "organism": organism,
        "n_terms": sum(1 for _ in open(path)),
        "created_at": datetime.now().strftime("%Y-%m-%d")
    })

# -----------------------------
# GO
# -----------------------------
add_entry(
    "GO_Biological_Process_2021",
    GO_DIR / "GO_Biological_Process_2021_fixed.gmt",
    "GO_BP",
    "Enrichr"
)

add_entry(
    "GO_Molecular_Function_2021",
    GO_DIR / "GO_Molecular_Function_2021_fixed.gmt",
    "GO_MF",
    "Enrichr"
)

add_entry(
    "GO_Cellular_Component_2021",
    GO_DIR / "GO_Cellular_Component_2021_fixed.gmt",
    "GO_CC",
    "Enrichr"
)

# -----------------------------
# KEGG
# -----------------------------
add_entry(
    "KEGG_2021_Human",
    KEGG_DIR / "KEGG_2021_Human_fixed.gmt",
    "KEGG",
    "Enrichr"
)

# -----------------------------
# Save
# -----------------------------
registry_df = pd.DataFrame(registry)

registry_path = META_DIR / "gene_sets_registry.csv"
registry_df.to_csv(registry_path, index=False)

print("[OK] Registry built")
print(registry_df)
'''

'\n# =========================================================\n# n3-01) Build Gene Set Registry (OFFLINE MODE)\n# =========================================================\n\nimport pandas as pd\nfrom datetime import datetime\n\nregistry = []\n\ndef add_entry(name, path, category, source, organism="human"):\n    registry.append({\n        "name": name,\n        "path": str(path),\n        "category": category,\n        "source": source,\n        "organism": organism,\n        "n_terms": sum(1 for _ in open(path)),\n        "created_at": datetime.now().strftime("%Y-%m-%d")\n    })\n\n# -----------------------------\n# GO\n# -----------------------------\nadd_entry(\n    "GO_Biological_Process_2021",\n    GO_DIR / "GO_Biological_Process_2021_fixed.gmt",\n    "GO_BP",\n    "Enrichr"\n)\n\nadd_entry(\n    "GO_Molecular_Function_2021",\n    GO_DIR / "GO_Molecular_Function_2021_fixed.gmt",\n    "GO_MF",\n    "Enrichr"\n)\n\nadd_entry(\n    "GO_Cellular_Component_2021",\n    GO_DIR / "GO_Cel

In [21]:
'''
# =========================================================
# n3-02) Gene Set Loader (OFFLINE)
# =========================================================

import pandas as pd

registry_path = META_DIR / "gene_sets_registry.csv"
registry_df = pd.read_csv(registry_path)

def get_gene_set(name=None, category=None):
    df = registry_df.copy()
    
    if name:
        df = df[df["name"] == name]
    if category:
        df = df[df["category"] == category]
    
    if df.empty:
        raise ValueError("No gene set found!")
    
    return df["path"].tolist()

# تست
print(get_gene_set(category="GO_BP"))
'''

'\n# =========================================================\n# n3-02) Gene Set Loader (OFFLINE)\n# =========================================================\n\nimport pandas as pd\n\nregistry_path = META_DIR / "gene_sets_registry.csv"\nregistry_df = pd.read_csv(registry_path)\n\ndef get_gene_set(name=None, category=None):\n    df = registry_df.copy()\n\n    if name:\n        df = df[df["name"] == name]\n    if category:\n        df = df[df["category"] == category]\n\n    if df.empty:\n        raise ValueError("No gene set found!")\n\n    return df["path"].tolist()\n\n# تست\nprint(get_gene_set(category="GO_BP"))\n'

In [22]:
'''
# =========================================================
# n3-03) Offline GO/KEGG Enrichment
# =========================================================

import gseapy as gp

def run_offline_enrichment(gene_list, category="GO_BP", top_n=20):
    
    gene_sets = get_gene_set(category=category)
    
    all_results = []
    
    for gs in gene_sets:
        print(f"[RUN] {gs}")
        
        enr = gp.enrich(
            gene_list=gene_list,
            gene_sets=gs,
            outdir=None
        )
        
        res = enr.results.copy()
        res["source_file"] = gs
        
        all_results.append(res)
    
    final_df = pd.concat(all_results, ignore_index=True)
    final_df = final_df.sort_values("Adjusted P-value").head(top_n)
    
    return final_df
'''    

'\n# =========================================================\n# n3-03) Offline GO/KEGG Enrichment\n# =========================================================\n\nimport gseapy as gp\n\ndef run_offline_enrichment(gene_list, category="GO_BP", top_n=20):\n\n    gene_sets = get_gene_set(category=category)\n\n    all_results = []\n\n    for gs in gene_sets:\n        print(f"[RUN] {gs}")\n\n        enr = gp.enrich(\n            gene_list=gene_list,\n            gene_sets=gs,\n            outdir=None\n        )\n\n        res = enr.results.copy()\n        res["source_file"] = gs\n\n        all_results.append(res)\n\n    final_df = pd.concat(all_results, ignore_index=True)\n    final_df = final_df.sort_values("Adjusted P-value").head(top_n)\n\n    return final_df\n'

In [23]:
'''
# =========================================================
# n3-04) Enrichment per cluster (FINAL FIXED)
# =========================================================

cluster_key = CLUSTER_KEY
results_all = []

print("[STEP] Running GO enrichment per cluster...")

for cl in adata.obs[cluster_key].unique():

    cl_int = int(cl)   # ← اصلاح اصلی

    genes = markers_df[
        (markers_df["group"] == cl_int) &
        (markers_df["pvals_adj"] < 0.05) &
        (markers_df["logfoldchanges"] > 0.25)
    ].sort_values("logfoldchanges", ascending=False)["names"].head(100).tolist()

    print(f"[INFO] Cluster {cl} ({cl_int}): {len(genes)} genes")

    if len(genes) < 5:
        print(f"[SKIP] Cluster {cl}: too few genes")
        continue

    try:
        enr = run_offline_enrichment(genes, category="GO_BP")

        if enr is None or enr.empty:
            print(f"[SKIP] Cluster {cl}: no enrichment results")
            continue

        enr["cluster"] = cl
        results_all.append(enr)

    except Exception as e:
        print(f"[ERROR] Cluster {cl}: {e}")
        continue

# -------------------------
# Safe concatenation
# -------------------------
if len(results_all) == 0:
    print("[WARNING] No enrichment results found for any cluster.")
else:
    final_enrichment = pd.concat(results_all, ignore_index=True)

    save_path = RESULTS_DIR / f"GO_enrichment_clusters.csv"
    final_enrichment.to_csv(save_path, index=False)

    print(f"[OK] Saved → {save_path}")
'''    

'\n# =========================================================\n# n3-04) Enrichment per cluster (FINAL FIXED)\n# =========================================================\n\ncluster_key = CLUSTER_KEY\nresults_all = []\n\nprint("[STEP] Running GO enrichment per cluster...")\n\nfor cl in adata.obs[cluster_key].unique():\n\n    cl_int = int(cl)   # ← اصلاح اصلی\n\n    genes = markers_df[\n        (markers_df["group"] == cl_int) &\n        (markers_df["pvals_adj"] < 0.05) &\n        (markers_df["logfoldchanges"] > 0.25)\n    ].sort_values("logfoldchanges", ascending=False)["names"].head(100).tolist()\n\n    print(f"[INFO] Cluster {cl} ({cl_int}): {len(genes)} genes")\n\n    if len(genes) < 5:\n        print(f"[SKIP] Cluster {cl}: too few genes")\n        continue\n\n    try:\n        enr = run_offline_enrichment(genes, category="GO_BP")\n\n        if enr is None or enr.empty:\n            print(f"[SKIP] Cluster {cl}: no enrichment results")\n            continue\n\n        enr["cluster"] =

In [24]:
# =============================================================================
# n3-16 + n3-17) OFFLINE GO Enrichment (FINAL STABLE VERSION)
# =============================================================================

import pandas as pd
import gseapy as gp

print("\n[STEP] Starting OFFLINE GO enrichment pipeline...")

# =========================================================
# CONFIG (offline paths)
# =========================================================

GO_PATHS = {
    "GO_BP": GO_DIR / "GO_Biological_Process_2021_fixed.gmt",
    "GO_MF": GO_DIR / "GO_Molecular_Function_2021_fixed.gmt",
    "GO_CC": GO_DIR / "GO_Cellular_Component_2021_fixed.gmt",
}

TOP_N = 100
PVAL_THRESH = 0.05
LOGFC_THRESH = 0.25

# =========================================================
# Helper: run offline enrichment
# =========================================================

def run_offline_go(genes, gmt_path, group_name, category):

    print(f"[GO - {group_name}] Running {category} using:")
    print(f"   → {gmt_path}")

    try:
        enr = gp.enrich(
            gene_list=genes,
            gene_sets=str(gmt_path),
            outdir=None,
            cutoff=0.05
        )

    except Exception as e:
        print(f"[GO - {group_name}] ❌ ERROR ({category}): {e}")
        return None

    if enr is None or not hasattr(enr, "results") or enr.results is None:
        print(f"[GO - {group_name}] ❌ No result object ({category})")
        return None

    if enr.results.empty:
        print(f"[GO - {group_name}] ⚠️ No enrichment ({category})")
        return None

    res = enr.results.copy()

    res["group"] = group_name
    res["category"] = category
    res["n_genes"] = len(genes)

    print(f"[GO - {group_name}] ✅ {category}: {res.shape[0]} terms")

    return res

# =========================================================
# Detect DE structure
# =========================================================

if "markers_df" not in globals() or markers_df is None or markers_df.empty:
    raise ValueError("[FATAL] markers_df not found or empty")

print(f"[INFO] markers_df shape: {markers_df.shape}")

if {"group", "names", "pvals_adj"}.issubset(markers_df.columns):
    group_col = "group"
    gene_col = "names"
    pval_col = "pvals_adj"
    print("[DE] Using cluster markers")

elif {"celltype", "names", "pvals_adj"}.issubset(markers_df.columns):
    group_col = "celltype"
    gene_col = "names"
    pval_col = "pvals_adj"
    print("[DE] Using cell-type markers")

else:
    raise ValueError(f"[FATAL] Cannot detect DE structure. Columns: {list(markers_df.columns)}")

groups = sorted(markers_df[group_col].dropna().unique())
print(f"[INFO] Groups detected: {groups}")

# =========================================================
# Run enrichment per group
# =========================================================

all_results = []

for g in groups:

    print("\n" + "="*70)
    print(f"[RUN] Group: {g}")
    print("="*70)

    df = markers_df[markers_df[group_col] == g].copy()

    # -------------------------
    # Filtering
    # -------------------------
    df = df[df[pval_col] < PVAL_THRESH]

    if "logfoldchanges" in df.columns:
        df = df[df["logfoldchanges"] > LOGFC_THRESH]

    df = df.sort_values("logfoldchanges", ascending=False).head(TOP_N)

    genes = df[gene_col].dropna().astype(str).unique().tolist()

    print(f"[INFO] Selected genes: {len(genes)}")

    if len(genes) < 10:
        print(f"[SKIP] Too few genes")
        continue

    # -------------------------
    # Run all GO categories
    # -------------------------
    for cat, gmt_path in GO_PATHS.items():

        if not gmt_path.exists():
            print(f"[ERROR] Missing file: {gmt_path}")
            continue

        res = run_offline_go(genes, gmt_path, str(g), cat)

        if res is not None:
            all_results.append(res)

            out_individual = RESULTS_DIR / f"12-{DATASET_NAME}_GO_{cat}_group_{g}.csv"
            res.to_csv(out_individual, index=False)

            print(f"[SAVE] {out_individual}")

# =========================================================
# Combine results
# =========================================================

print("\n[STEP] Combining results...")

if len(all_results) == 0:
    print("[WARNING] No enrichment results generated.")

else:
    final_df = pd.concat(all_results, ignore_index=True)

    if "Adjusted P-value" in final_df.columns:
        final_df = final_df.sort_values(["group", "Adjusted P-value"])

    # Global FDR
    try:
        from statsmodels.stats.multitest import multipletests

        if "Adjusted P-value" in final_df.columns:
            final_df["p_adj_global"] = multipletests(
                final_df["Adjusted P-value"],
                method="fdr_bh"
            )[1]

            print("[OK] Global FDR added")

    except ImportError:
        print("[WARNING] statsmodels not installed")

    out_final = RESULTS_DIR / f"13-{DATASET_NAME}_GO_ALL_OFFLINE.csv"
    final_df.to_csv(out_final, index=False)

    print("\n" + "="*60)
    print(f"[FINAL OUTPUT] {out_final}")
    print("="*60)

    print("\n[SUMMARY]")
    print(final_df.groupby(["group", "category"]).size())


[STEP] Starting OFFLINE GO enrichment pipeline...
[INFO] markers_df shape: (81936, 8)
[DE] Using cluster markers
[INFO] Groups detected: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

[RUN] Group: 0
[INFO] Selected genes: 100
[GO - 0] Running GO_BP using:
   → K:\@scRNA-1.1-PBMC3K\resources\go\GO_Biological_Process_2021_fixed.gmt
[GO - 0] ✅ GO_BP: 939 terms
[SAVE] K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\12-260504-pbmc3k_GO_GO_BP_group_0.csv
[GO - 0] Running GO_MF using:
   → K:\@scRNA-1.1-PBMC3K\resources\go\GO_Molecular_Function_2021_fixed.gmt
[GO - 0] ✅ GO_MF: 154 terms
[SAVE] K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\12-260504-pbmc3k_GO_GO_MF_group_0.csv
[GO - 0] Running GO_CC using:
   → K:\@scRNA-1.1-PBMC3K\resources\go\GO_Cellular_Component_2021_fixed.gmt
[GO - 0] ✅ GO_CC: 59 terms
[SAVE] K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\12-260504-pbmc3k_GO_GO_CC_group_0.csv

[RUN] Group: 1
[INFO] Selected genes: 100
[GO - 1] Running GO_BP using:
  

In [26]:
# =============================================================================
# n3-18) Advanced GO Visualization (FINAL FIXED FOR OFFLINE PIPELINE)
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

print("[STEP 18] GO visualization started...")

# =========================================================
# Load GO results
# =========================================================

go_file = RESULTS_DIR / f"13-{DATASET_NAME}_GO_ALL_OFFLINE.csv"

if not go_file.exists():
    raise FileNotFoundError(f"[ERROR] File not found: {go_file}")

go_all_df = pd.read_csv(go_file)

if go_all_df.empty:
    print("[INFO] GO table empty → skipping")
else:

    print(f"[INFO] GO table size: {go_all_df.shape}")
    print(f"[INFO] Columns: {list(go_all_df.columns)}")

    # =========================================================
    # Preprocessing
    # =========================================================

    go_all_df["log10_padj"] = -np.log10(go_all_df["Adjusted P-value"] + 1e-300)

    # -------- FIX: use 'category' instead of go_library
    def simplify_category(x):
        x = str(x)
        if "BP" in x:
            return "BP"
        elif "MF" in x:
            return "MF"
        elif "CC" in x:
            return "CC"
        else:
            return "Other"

    if "category" in go_all_df.columns:
        go_all_df["GO_cat"] = go_all_df["category"].apply(simplify_category)
    else:
        print("[WARNING] 'category' column not found → assigning 'Other'")
        go_all_df["GO_cat"] = "Other"

    # -------- gene count
    def extract_gene_count(x):
        try:
            return int(str(x).split("/")[0])
        except:
            return 1

    if "Overlap" in go_all_df.columns:
        go_all_df["gene_count"] = go_all_df["Overlap"].apply(extract_gene_count)
    else:
        go_all_df["gene_count"] = 1

    # -------- shorten GO terms
    def shorten(term, max_len=55):
        term = str(term)
        return term if len(term) <= max_len else term[:max_len-3] + "..."

    # =========================================================
    # Colors
    # =========================================================

    color_map = {
        "BP": "#1f77b4",
        "MF": "#ff7f0e",
        "CC": "#2ca02c",
        "Other": "gray"
    }

    legend_elements = [
        Line2D([0], [0], marker='o', color='w',
               label=cat,
               markerfacecolor=color_map[cat],
               markersize=8)
        for cat in ["BP", "MF", "CC"]
    ]

    groups = sorted(go_all_df["group"].unique())
    print(f"[INFO] Groups: {groups}")

    fig_counter = 20

    # =========================================================
    # 1. Per-group bubble plots
    # =========================================================

    for g in groups:

        df_g = go_all_df[go_all_df["group"] == g].copy()

        if df_g.empty:
            continue

        df_g = df_g.sort_values("Adjusted P-value").head(10)
        df_g = df_g.iloc[::-1]
        df_g["Term_short"] = df_g["Term"].apply(shorten)

        print(f"[PLOT] Group {g}")

        plt.figure(figsize=(7, 4))

        plt.scatter(
            df_g["log10_padj"],
            range(len(df_g)),
            s=np.sqrt(df_g["gene_count"]) * 40,
            c=df_g["GO_cat"].map(color_map),
            alpha=0.85
        )

        plt.yticks(range(len(df_g)), df_g["Term_short"])
        plt.xlabel("-log10(adj p-value)")
        plt.title(f"GO Enrichment (Cluster {g})")

        plt.legend(handles=legend_elements, title="Category")

        path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GO_cluster_{g}.png"
        plt.savefig(path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"[FIG] {path}")
        fig_counter += 1

    # =========================================================
    # 2. Combined plot
    # =========================================================

    print("[PLOT] Combined GO plot")

    df_top = (
        go_all_df
        .sort_values("Adjusted P-value")
        .groupby("group")
        .head(5)
        .copy()
    )

    df_top["Term_short"] = df_top["Term"].apply(shorten)

    plt.figure(figsize=(9, 5))

    plt.scatter(
        df_top["group"].astype(str),
        df_top["Term_short"],
        s=np.sqrt(df_top["gene_count"]) * 40,
        c=df_top["GO_cat"].map(color_map),
        alpha=0.8
    )

    plt.xlabel("Cluster")
    plt.ylabel("GO Term")
    plt.title("Top GO Terms Across Clusters")

    plt.legend(handles=legend_elements, title="Category")

    path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GO_combined.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"[FIG] {path}")
    fig_counter += 1

    # =========================================================
    # 3. Category barplots
    # =========================================================

    for cat in ["BP", "MF", "CC"]:

        df_cat = go_all_df[go_all_df["GO_cat"] == cat].copy()

        if df_cat.empty:
            continue

        df_cat = df_cat.sort_values("Adjusted P-value").head(15)
        df_cat["Term_short"] = df_cat["Term"].apply(shorten)

        plt.figure(figsize=(7, 5))

        plt.barh(
            df_cat["Term_short"],
            df_cat["log10_padj"]
        )

        plt.xlabel("-log10(adj p-value)")
        plt.title(f"Top {cat} Terms")

        plt.gca().invert_yaxis()

        path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_GO_{cat}.png"
        plt.savefig(path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"[FIG] {path}")
        fig_counter += 1

    print("\n[STEP 18 COMPLETED] GO visualization finished.")

[STEP 18] GO visualization started...
[INFO] GO table size: (7471, 12)
[INFO] Columns: ['Gene_set', 'Term', 'Overlap', 'P-value', 'Adjusted P-value', 'Odds Ratio', 'Combined Score', 'Genes', 'group', 'category', 'n_genes', 'p_adj_global']
[INFO] Groups: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
[PLOT] Group 0
[FIG] K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\20-260504-pbmc3k_GO_cluster_0.png
[PLOT] Group 1
[FIG] K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\21-260504-pbmc3k_GO_cluster_1.png
[PLOT] Group 2
[FIG] K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\22-260504-pbmc3k_GO_cluster_2.png
[PLOT] Group 3
[FIG] K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\23-260504-pbmc3k_GO_cluster_3.png
[PLOT] Group 4
[FIG] K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\24-260504-pbmc3k_GO_cluster_4.png
[PLOT] Group 5
[FIG] K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\25-260504-pbmc3k_GO_cluster_5.png
[PLOT] Combined GO plot
[FIG] K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\26-260

In [27]:
##  End of notebook 3 - GO Enrichment Analysis (BP, MF, CC) 

In [28]:
## ---------------------------------------------

In [29]:
## Notebook 4 - pathway analysis (KEGG / Reactome)

In [30]:
'''
# =============================================================================
# FIX Reactome (ROBUST VERSION - handles multiple formats)
# =============================================================================

from pathlib import Path

INPUT_FILE = RESOURCE_DIR / "reactome" / "Reactome_2022.txt"
OUTPUT_FILE = RESOURCE_DIR / "reactome" / "Reactome_2022_fixed.gmt"

print(f"[INPUT] {INPUT_FILE}")
print(f"[OUTPUT] {OUTPUT_FILE}")

valid_lines = 0
total_lines = 0

with open(INPUT_FILE, "r", encoding="utf-8", errors="ignore") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line in fin:
        total_lines += 1

        line = line.strip()
        if not line:
            continue

        parts = line.split("\t")

        # باید حداقل 3 ستون داشته باشد
        if len(parts) < 3:
            continue

        # -----------------------------
        # تشخیص ساختار
        # -----------------------------
        if parts[0].startswith("R-HSA"):
            # حالت B
            pathway_id = parts[0]
            pathway_name = parts[1]
            genes = parts[2:]
            term = f"{pathway_name} ({pathway_id})"
        else:
            # حالت A
            term = parts[0]
            genes = parts[1:]

        # پاکسازی ژن‌ها
        genes = [g.strip() for g in genes if g.strip() != ""]

        if len(genes) < 5:
            continue

        # ساخت GMT استاندارد
        new_line = "\t".join([term, "na"] + genes)

        fout.write(new_line + "\n")
        valid_lines += 1

print("\n[RESULT]")
print(f"  Total lines: {total_lines}")
print(f"  Valid lines: {valid_lines}")
print(f"  Saved to: {OUTPUT_FILE}")
'''

'\n# =============================================================================\n# FIX Reactome (ROBUST VERSION - handles multiple formats)\n# =============================================================================\n\nfrom pathlib import Path\n\nINPUT_FILE = RESOURCE_DIR / "reactome" / "Reactome_2022.txt"\nOUTPUT_FILE = RESOURCE_DIR / "reactome" / "Reactome_2022_fixed.gmt"\n\nprint(f"[INPUT] {INPUT_FILE}")\nprint(f"[OUTPUT] {OUTPUT_FILE}")\n\nvalid_lines = 0\ntotal_lines = 0\n\nwith open(INPUT_FILE, "r", encoding="utf-8", errors="ignore") as fin,      open(OUTPUT_FILE, "w", encoding="utf-8") as fout:\n\n    for line in fin:\n        total_lines += 1\n\n        line = line.strip()\n        if not line:\n            continue\n\n        parts = line.split("\t")\n\n        # باید حداقل 3 ستون داشته باشد\n        if len(parts) < 3:\n            continue\n\n        # -----------------------------\n        # تشخیص ساختار\n        # -----------------------------\n        if parts[0].s

In [40]:
# =============================================================================
# n4-19) Pathway setup (OFFLINE VERSION: KEGG + Reactome)
# =============================================================================

import pandas as pd
import numpy as np
import gseapy as gp
from pathlib import Path

print("[STEP n4-19] Initializing OFFLINE Pathway Enrichment environment...")

# -----------------------------------------------------------------------------
# 19-1. Define OFFLINE gene set paths (REAL PATHS)
# -----------------------------------------------------------------------------
KEGG_GMT = KEGG_DIR / "KEGG_2021_Human_fixed.gmt"
REACTOME_GMT = RESOURCE_DIR / "reactome" / "Reactome_2022_fixed.gmt"

PATHWAY_GENE_SETS = {
    "KEGG": KEGG_GMT,
    "REACTOME": REACTOME_GMT
}

print("\n[INFO] Pathway gene sets (OFFLINE):")
for k, v in PATHWAY_GENE_SETS.items():
    print(f"  {k}: {v}")

# -----------------------------------------------------------------------------
# 19-2. Validate files exist
# -----------------------------------------------------------------------------
VALID_GENESETS = {}

for name, path in PATHWAY_GENE_SETS.items():
    if Path(path).exists():
        VALID_GENESETS[name] = path
        print(f"[OK] Found: {path}")
    else:
        print(f"[ERROR] Missing: {path}")

# -----------------------------------------------------------------------------
# 19-3. Parameters
# -----------------------------------------------------------------------------
PVAL_THRESH_PATHWAY = 0.05
TOP_N_PATHWAY = 100
ORGANISM_PATHWAY = adata.uns.get("organism", "human")

print(f"\n[INFO] Organism: {ORGANISM_PATHWAY}")
print(f"[INFO] p-value threshold: {PVAL_THRESH_PATHWAY}")
print(f"[INFO] top N genes per group: {TOP_N_PATHWAY}")

# -----------------------------------------------------------------------------
# 19-4. Check markers_df
# -----------------------------------------------------------------------------
try:
    markers_df
except NameError:
    print("[ERROR] markers_df not found")
    PATHWAY_READY = False
else:
    print(f"[INFO] markers_df detected → shape: {markers_df.shape}")
    PATHWAY_READY = True

# -----------------------------------------------------------------------------
# 19-5. Detect structure
# -----------------------------------------------------------------------------
if PATHWAY_READY:

    if {"group", "names", "pvals_adj"}.issubset(markers_df.columns):
        PATHWAY_GROUP_COL = "group"
        PATHWAY_GENE_COL = "names"
        PATHWAY_PVAL_COL = "pvals_adj"
        print("[INFO] Using cluster markers format")

    elif {"celltype", "names", "pvals_adj"}.issubset(markers_df.columns):
        PATHWAY_GROUP_COL = "celltype"
        PATHWAY_GENE_COL = "names"
        PATHWAY_PVAL_COL = "pvals_adj"
        print("[INFO] Using cell-type markers format")

    else:
        print("[ERROR] Unknown DE structure")
        PATHWAY_READY = False

# -----------------------------------------------------------------------------
# 19-6. Groups
# -----------------------------------------------------------------------------
if PATHWAY_READY:
    PATHWAY_GROUPS = sorted(markers_df[PATHWAY_GROUP_COL].unique())
    print(f"[INFO] Groups detected: {PATHWAY_GROUPS}")
else:
    PATHWAY_GROUPS = []

# -----------------------------------------------------------------------------
# 19-7. Final summary
# -----------------------------------------------------------------------------
print("\n[SUMMARY n4-19]")
print(f"  Pathway ready: {PATHWAY_READY}")
print(f"  Gene sets available: {list(VALID_GENESETS.keys())}")
print(f"  Groups detected: {len(PATHWAY_GROUPS)}")

print("[STEP n4-19 COMPLETED]")

[STEP n4-19] Initializing OFFLINE Pathway Enrichment environment...

[INFO] Pathway gene sets (OFFLINE):
  KEGG: K:\@scRNA-1.1-PBMC3K\resources\kegg\KEGG_2021_Human_fixed.gmt
  REACTOME: K:\@scRNA-1.1-PBMC3K\resources\reactome\Reactome_2022_fixed.gmt
[OK] Found: K:\@scRNA-1.1-PBMC3K\resources\kegg\KEGG_2021_Human_fixed.gmt
[OK] Found: K:\@scRNA-1.1-PBMC3K\resources\reactome\Reactome_2022_fixed.gmt

[INFO] Organism: human
[INFO] p-value threshold: 0.05
[INFO] top N genes per group: 100
[INFO] markers_df detected → shape: (81936, 8)
[INFO] Using cluster markers format
[INFO] Groups detected: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

[SUMMARY n4-19]
  Pathway ready: True
  Gene sets available: ['KEGG', 'REACTOME']
  Groups detected: 6
[STEP n4-19 COMPLETED]


In [41]:
# =============================================================================
# n4-20) Helper: Run OFFLINE Pathway enrichment for one group
# =============================================================================

def run_pathway_enrichment_for_group(
    gene_table,
    group_name,
    gene_col=PATHWAY_GENE_COL,
    pval_col=PATHWAY_PVAL_COL,
    pval_thresh=PVAL_THRESH_PATHWAY,
    top_n=TOP_N_PATHWAY,
    organism=ORGANISM_PATHWAY,
    gene_sets_dict=VALID_GENESETS,
    verbose=True
):

    print("\n" + "="*70)
    print(f"[PATHWAY - {group_name}] START")
    print("="*70)

    # -------------------------------------------------------------------------
    # Input check
    # -------------------------------------------------------------------------
    if gene_table is None or gene_table.empty:
        print("[ERROR] Empty gene table")
        return None

    # -------------------------------------------------------------------------
    # Filtering
    # -------------------------------------------------------------------------
    df = gene_table.sort_values(pval_col, ascending=True)

    if "logfoldchanges" in df.columns:
        df = df.query(f"{pval_col} <= @pval_thresh and logfoldchanges > 0")
    else:
        df = df.query(f"{pval_col} <= @pval_thresh")

    df = df.head(top_n)

    genes = df[gene_col].dropna().astype(str).unique().tolist()

    print(f"[INFO] Genes used: {len(genes)}")

    if len(genes) < 10:
        print("[SKIP] Too few genes")
        return None

    results_all = []

    # -------------------------------------------------------------------------
    # Run each database separately
    # -------------------------------------------------------------------------
    for db_name, gmt_path in gene_sets_dict.items():

        print(f"\n[RUN] {db_name}")
        print(f"   → {gmt_path}")

        try:
            enr = gp.enrichr(
                gene_list=genes,
                gene_sets=str(gmt_path),
                organism=organism,
                outdir=None,
                cutoff=0.05
            )

            if enr is None or enr.results is None or enr.results.empty:
                print(f"[SKIP] {db_name} → no results")
                continue

            res = enr.results.copy()
            res["group"] = group_name
            res["pathway_db"] = db_name
            res["n_genes"] = len(genes)

            results_all.append(res)

            print(f"[OK] {db_name}: {res.shape[0]} pathways")

        except Exception as e:
            print(f"[ERROR] {db_name}: {e}")
            continue

    if not results_all:
        print("[WARNING] No pathway results")
        return None

    return pd.concat(results_all, ignore_index=True)

In [42]:
# =============================================================================
# n4-21) Run OFFLINE Pathway enrichment for all groups
# =============================================================================

print("[STEP n4-21] Running OFFLINE pathway enrichment...")

if not PATHWAY_READY:
    print("[ERROR] Pathway environment not ready")

elif len(VALID_GENESETS) == 0:
    print("[ERROR] No valid gene sets found")

else:

    all_pathway_results = []

    print(f"[INFO] Total groups: {len(PATHWAY_GROUPS)}")

    for g_name in PATHWAY_GROUPS:

        print("\n" + "-"*80)
        print(f"[RUN] Group: {g_name}")
        print("-"*80)

        subset_df = markers_df[
            markers_df[PATHWAY_GROUP_COL] == g_name
        ].copy()

        pathway_res = run_pathway_enrichment_for_group(
            gene_table=subset_df,
            group_name=str(g_name)
        )

        if pathway_res is not None and not pathway_res.empty:

            all_pathway_results.append(pathway_res)

            out_file = RESULTS_DIR / f"14-{DATASET_NAME}_Pathway_group_{g_name}.csv"
            pathway_res.to_csv(out_file, index=False)

            print(f"[SAVE] {out_file}")

        else:
            print(f"[INFO] No results for group {g_name}")

    # -------------------------------------------------------------------------
    # Combine
    # -------------------------------------------------------------------------
    if all_pathway_results:

        pathway_all_df = pd.concat(all_pathway_results, ignore_index=True)

        print(f"\n[INFO] Combined size: {pathway_all_df.shape}")

        if "Adjusted P-value" in pathway_all_df.columns:
            pathway_all_df = pathway_all_df.sort_values(
                ["group", "Adjusted P-value"]
            )

        # Global FDR
        try:
            from statsmodels.stats.multitest import multipletests

            if "Adjusted P-value" in pathway_all_df.columns:
                pathway_all_df["p_adj_global"] = multipletests(
                    pathway_all_df["Adjusted P-value"],
                    method="fdr_bh"
                )[1]

                print("[OK] Global FDR added")

        except ImportError:
            print("[WARNING] statsmodels not installed")

        out_all = RESULTS_DIR / f"15-{DATASET_NAME}_Pathway_ALL_OFFLINE.csv"
        pathway_all_df.to_csv(out_all, index=False)

        print("\n" + "="*80)
        print(f"[FINAL OUTPUT] {out_all}")
        print("="*80)

        print("\n[SUMMARY]")
        print(pathway_all_df.groupby(["group", "pathway_db"]).size())

    else:
        print("[WARNING] No pathway results generated")

[STEP n4-21] Running OFFLINE pathway enrichment...
[INFO] Total groups: 6

--------------------------------------------------------------------------------
[RUN] Group: 0
--------------------------------------------------------------------------------

[PATHWAY - 0] START
[INFO] Genes used: 100

[RUN] KEGG
   → K:\@scRNA-1.1-PBMC3K\resources\kegg\KEGG_2021_Human_fixed.gmt
[OK] KEGG: 106 pathways

[RUN] REACTOME
   → K:\@scRNA-1.1-PBMC3K\resources\reactome\Reactome_2022_fixed.gmt
[OK] REACTOME: 493 pathways
[SAVE] K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\14-260504-pbmc3k_Pathway_group_0.csv

--------------------------------------------------------------------------------
[RUN] Group: 1
--------------------------------------------------------------------------------

[PATHWAY - 1] START
[INFO] Genes used: 100

[RUN] KEGG
   → K:\@scRNA-1.1-PBMC3K\resources\kegg\KEGG_2021_Human_fixed.gmt
[OK] KEGG: 74 pathways

[RUN] REACTOME
   → K:\@scRNA-1.1-PBMC3K\resources\reactome\Reactome_2022_fi

In [ ]:
'''
# test
test_df = markers_df.head(200)

res = run_pathway_enrichment_for_group(
    gene_table=test_df,
    group_name="TEST"
)

print(type(res))
'''

In [43]:
# =============================================================================
# n4-22) Pathway Visualization (KEGG + Reactome) — ROBUST FINAL
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D

print("\n[STEP n4-22] Pathway visualization started...\n")

# -----------------------------------------------------------------------------
# 22-0. Check input
# -----------------------------------------------------------------------------
try:
    pathway_all_df
except NameError:
    print("[ERROR] pathway_all_df not found. Run n4-21 first.")
else:

    if pathway_all_df.empty:
        print("[INFO] Pathway table is empty. Skipping plots.")
    else:

        print(f"[INFO] Table size: {pathway_all_df.shape}")
        print(f"[INFO] Columns: {list(pathway_all_df.columns)}")

        if "Adjusted P-value" not in pathway_all_df.columns:
            print("[ERROR] 'Adjusted P-value' column missing.")
        else:

            # -----------------------------------------------------------------------------
            # 22-1. Preprocessing
            # -----------------------------------------------------------------------------
            pathway_all_df["log10_padj"] = -np.log10(
                pathway_all_df["Adjusted P-value"] + 1e-300
            )

            # -------- Detect pathway source column --------
            if "pathway_db" in pathway_all_df.columns:
                source_col = "pathway_db"
            elif "Gene_set" in pathway_all_df.columns:
                source_col = "Gene_set"
            else:
                source_col = None

            # -------- Extract pathway type --------
            def extract_pathway_type(x):
                x = str(x)
                if "KEGG" in x.upper():
                    return "KEGG"
                elif "REACTOME" in x.upper():
                    return "Reactome"
                return "Other"

            if source_col:
                pathway_all_df["pathway_type"] = pathway_all_df[source_col].apply(extract_pathway_type)
            else:
                pathway_all_df["pathway_type"] = "Other"

            # -------- Gene count --------
            def extract_gene_count(x):
                try:
                    return int(str(x).split("/")[0])
                except:
                    return 1

            if "Overlap" in pathway_all_df.columns:
                pathway_all_df["gene_count"] = pathway_all_df["Overlap"].apply(extract_gene_count)
            else:
                pathway_all_df["gene_count"] = 1

            # -------- Shorten pathway names --------
            def shorten(term, max_len=60):
                term = str(term)
                return term if len(term) <= max_len else term[:max_len-3] + "..."

            pathway_all_df["Term_short"] = pathway_all_df["Term"].apply(shorten)

            # -----------------------------------------------------------------------------
            # Colors & legend
            # -----------------------------------------------------------------------------
            color_map = {
                "KEGG": "tab:red",
                "Reactome": "tab:blue",
                "Other": "gray"
            }

            legend_elements = [
                Line2D([0], [0], marker='o', color='w',
                       label=cat,
                       markerfacecolor=color_map[cat],
                       markersize=8)
                for cat in ["KEGG", "Reactome"]
            ]

            groups = sorted(pathway_all_df["group"].unique())
            print(f"[INFO] Groups: {groups}")

            fig_counter = 30

            # -----------------------------------------------------------------------------
            # 22-2. Per-group bubble plots
            # -----------------------------------------------------------------------------
            for g in groups:

                df_g = pathway_all_df[pathway_all_df["group"] == g].copy()

                if df_g.empty:
                    continue

                df_g = df_g.sort_values("Adjusted P-value").head(10)
                df_g = df_g.iloc[::-1]

                print(f"[PLOT] Group {g} → {len(df_g)} pathways")

                plt.figure(figsize=(7, 4))

                plt.scatter(
                    df_g["log10_padj"],
                    range(len(df_g)),
                    s=df_g["gene_count"] * 25,
                    c=df_g["pathway_type"].map(color_map),
                    alpha=0.8
                )

                plt.yticks(range(len(df_g)), df_g["Term_short"])
                plt.xlabel("-log10(adj p-value)")
                plt.title(f"Pathway Enrichment (Group {g})")

                plt.legend(handles=legend_elements, title="Database")

                out_path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_PATHWAY_bubble_{g}.png"
                plt.savefig(out_path, dpi=300, bbox_inches="tight")
                plt.close()

                print(f"[FIG] Saved → {out_path}")
                fig_counter += 1

            # -----------------------------------------------------------------------------
            # 22-3. Combined plot
            # -----------------------------------------------------------------------------
            print("[PLOT] Combined pathway plot")

            df_top = (
                pathway_all_df
                .sort_values("Adjusted P-value")
                .groupby("group")
                .head(5)
            )

            plt.figure(figsize=(9, 5))

            plt.scatter(
                df_top["group"],
                df_top["Term_short"],
                s=df_top["gene_count"] * 25,
                c=df_top["pathway_type"].map(color_map),
                alpha=0.7
            )

            plt.xlabel("Group")
            plt.ylabel("Pathway")
            plt.title("Pathway Enrichment Across Groups")

            plt.legend(handles=legend_elements, title="Database")

            combined_path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_PATHWAY_combined.png"
            plt.savefig(combined_path, dpi=300, bbox_inches="tight")
            plt.close()

            print(f"[FIG] Combined plot saved → {combined_path}")
            fig_counter += 1

            # -----------------------------------------------------------------------------
            # 22-4. Barplots per database
            # -----------------------------------------------------------------------------
            for db in ["KEGG", "Reactome"]:

                df_db = pathway_all_df[pathway_all_df["pathway_type"] == db]

                if df_db.empty:
                    continue

                df_db = df_db.sort_values("Adjusted P-value").head(15)

                print(f"[PLOT] {db} barplot")

                plt.figure(figsize=(7, 5))

                plt.barh(
                    df_db["Term_short"],
                    df_db["log10_padj"]
                )

                plt.xlabel("-log10(adj p-value)")
                plt.title(f"Top {db} Pathways")

                db_path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_PATHWAY_{db}.png"
                plt.savefig(db_path, dpi=300, bbox_inches="tight")
                plt.close()

                print(f"[FIG] Saved → {db_path}")
                fig_counter += 1

            # -----------------------------------------------------------------------------
            # 22-5. PDF report
            # -----------------------------------------------------------------------------
            pdf_path = FIG_DIR / f"{fig_counter}-{DATASET_NAME}_PATHWAY_report.pdf"

            with PdfPages(pdf_path) as pdf:

                for g in groups:

                    df_g = pathway_all_df[pathway_all_df["group"] == g]

                    if df_g.empty:
                        continue

                    df_g = df_g.sort_values("Adjusted P-value").head(10)

                    plt.figure(figsize=(8, 6))

                    plt.scatter(
                        df_g["log10_padj"],
                        range(len(df_g)),
                        s=df_g["gene_count"] * 25,
                        c=df_g["pathway_type"].map(color_map),
                        alpha=0.8
                    )

                    plt.yticks(range(len(df_g)), df_g["Term_short"])
                    plt.title(f"Pathway Enrichment (Group {g})")

                    plt.legend(handles=legend_elements, title="Database")

                    pdf.savefig(bbox_inches="tight")
                    plt.close()

            print(f"[FIG] PDF report saved → {pdf_path}")

            print("\n[STEP n4-22 COMPLETED] Pathway visualization finished.\n")


[STEP n4-22] Pathway visualization started...

[INFO] Table size: (3219, 12)
[INFO] Columns: ['Gene_set', 'Term', 'Overlap', 'P-value', 'Adjusted P-value', 'Odds Ratio', 'Combined Score', 'Genes', 'group', 'pathway_db', 'n_genes', 'p_adj_global']
[INFO] Groups: ['0', '1', '2', '3', '4', '5']
[PLOT] Group 0 → 10 pathways
[FIG] Saved → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\30-260504-pbmc3k_PATHWAY_bubble_0.png
[PLOT] Group 1 → 10 pathways
[FIG] Saved → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\31-260504-pbmc3k_PATHWAY_bubble_1.png
[PLOT] Group 2 → 10 pathways
[FIG] Saved → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\32-260504-pbmc3k_PATHWAY_bubble_2.png
[PLOT] Group 3 → 10 pathways
[FIG] Saved → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\33-260504-pbmc3k_PATHWAY_bubble_3.png
[PLOT] Group 4 → 10 pathways
[FIG] Saved → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\34-260504-pbmc3k_PATHWAY_bubble_4.png
[PLOT] Group 5 → 10 pathways
[FIG] Saved → K:\@scRNA-1.1-PBMC3K\figures\260504-pbmc3k\

In [ ]:
## END of Notebook 4 - pathway analysis (KEGG / Reactome)

In [47]:
# =============================================================================
# n5-23) Integrated Biological Summary (FIXED VERSION)
# =============================================================================

import pandas as pd
import numpy as np

print("[STEP n5-23] Building integrated biological summary...")

# -----------------------------------------------------------------------------
# 23-0. Load inputs
# -----------------------------------------------------------------------------
try:
    go_all_df
    pathway_all_df
except NameError:
    print("[ERROR] Required data not found. Run GO and Pathway steps first.")
else:

    if go_all_df.empty or pathway_all_df.empty:
        print("[ERROR] One of the inputs is empty.")
    else:

        go_df = go_all_df.copy()
        pw_df = pathway_all_df.copy()

        print(f"[INFO] GO table: {go_df.shape}")
        print(f"[INFO] Pathway table: {pw_df.shape}")

        # -----------------------------------------------------------------------------
        # 23-1. 🔥 CRITICAL FIX: unify group type
        # -----------------------------------------------------------------------------
        go_df["group"] = go_df["group"].astype(str)
        pw_df["group"] = pw_df["group"].astype(str)

        # -----------------------------------------------------------------------------
        # 23-2. Select top terms
        # -----------------------------------------------------------------------------
        go_top = (
            go_df
            .sort_values("Adjusted P-value")
            .groupby("group")
            .head(5)
        )

        pw_top = (
            pw_df
            .sort_values("Adjusted P-value")
            .groupby("group")
            .head(5)
        )

        print(f"[INFO] GO top rows: {go_top.shape}")
        print(f"[INFO] Pathway top rows: {pw_top.shape}")

        # -----------------------------------------------------------------------------
        # 23-3. Build summary
        # -----------------------------------------------------------------------------
        summary_rows = []

        groups = sorted(set(go_df["group"]).union(set(pw_df["group"])))

        print(f"[INFO] Total groups: {len(groups)}")

        for g in groups:

            go_terms = go_top[go_top["group"] == g]["Term"].tolist()
            pw_terms = pw_top[pw_top["group"] == g]["Term"].tolist()

            row = {
                "group": g,
                "GO_terms_top5": " | ".join(go_terms),
                "Pathway_terms_top5": " | ".join(pw_terms),
                "n_GO_terms": len(go_terms),
                "n_Pathways": len(pw_terms)
            }

            summary_rows.append(row)

        summary_df = pd.DataFrame(summary_rows)

        print(f"[INFO] Summary table: {summary_df.shape}")

        # -----------------------------------------------------------------------------
        # 23-4. Save
        # -----------------------------------------------------------------------------
        out_file = RESULTS_DIR / f"16-{DATASET_NAME}_Integrated_GO_Pathway_summary.csv"
        summary_df.to_csv(out_file, index=False)

        print("\n" + "=" * 70)
        print(f"[OUTPUT] Integrated summary saved:")
        print(f"         {out_file}")
        print("=" * 70)

        # -----------------------------------------------------------------------------
        # 23-5. Display preview
        # -----------------------------------------------------------------------------
        print("\n[PREVIEW]")
        display(summary_df.head())

        print("\n[STEP n5-23 COMPLETED]")

[STEP n5-23] Building integrated biological summary...
[INFO] GO table: (7471, 15)
[INFO] Pathway table: (3219, 16)
[INFO] GO top rows: (30, 15)
[INFO] Pathway top rows: (30, 16)
[INFO] Total groups: 6
[INFO] Summary table: (6, 5)

[OUTPUT] Integrated summary saved:
         K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\16-260504-pbmc3k_Integrated_GO_Pathway_summary.csv

[PREVIEW]


,group,GO_terms_top5,Pathway_terms_top5,n_GO_terms,n_Pathways
0,0,T cell activation (GO:0042110) | positive regu...,Eukaryotic Translation Elongation R-HSA-156842...,5,5
1,1,MHC class II protein complex (GO:0042613) | MH...,Eukaryotic Translation Elongation R-HSA-156842...,5,5
2,2,neutrophil degranulation (GO:0043312) | neutro...,Immune System R-HSA-168256 | Neutrophil Degran...,5,5
3,3,regulation of immune response (GO:0050776) | T...,Immunoregulatory Interactions Between A Lympho...,5,5
4,4,neutrophil activation involved in immune respo...,Immune System R-HSA-168256 | Neutrophil Degran...,5,5



[STEP n5-23 COMPLETED]


In [49]:
# =============================================================================
# n5-24) Semi-Automatic Cluster Annotation (ROBUST VERSION)
# =============================================================================

import pandas as pd

print("\n[STEP n5-24] Generating cluster annotations...\n")

# -----------------------------------------------------------------------------
# 24-0. Check input
# -----------------------------------------------------------------------------
try:
    summary_df
except NameError:
    print("[ERROR] summary_df not found. Run n5-23 first.")
else:

    if summary_df.empty:
        print("[ERROR] summary_df is empty.")
    else:

        print(f"[INFO] Summary columns: {list(summary_df.columns)}")

        # -----------------------------------------------------------------------------
        # 24-1. Detect column names dynamically (🔥 FIX)
        # -----------------------------------------------------------------------------
        if "GO_terms_top5" in summary_df.columns:
            GO_COL = "GO_terms_top5"
        elif "top_GO_terms" in summary_df.columns:
            GO_COL = "top_GO_terms"
        else:
            raise ValueError("GO terms column not found.")

        if "Pathway_terms_top5" in summary_df.columns:
            PW_COL = "Pathway_terms_top5"
        elif "top_pathways" in summary_df.columns:
            PW_COL = "top_pathways"
        else:
            raise ValueError("Pathway terms column not found.")

        print(f"[INFO] Using GO column: {GO_COL}")
        print(f"[INFO] Using Pathway column: {PW_COL}")

        # -----------------------------------------------------------------------------
        # 24-2. Annotation rules
        # -----------------------------------------------------------------------------
        def suggest_label(go_text, pathway_text):

            text = (str(go_text) + " " + str(pathway_text)).lower()

            # --- T cells ---
            if any(k in text for k in ["t cell", "tcr", "lymphocyte"]):
                return "T-cell / Lymphocyte"

            # --- B cells ---
            if any(k in text for k in ["b cell", "immunoglobulin"]):
                return "B-cell"

            # --- myeloid ---
            if any(k in text for k in ["monocyte", "macrophage", "myeloid"]):
                return "Myeloid"

            # --- inflammation ---
            if any(k in text for k in ["cytokine", "inflammatory", "immune response"]):
                return "Inflammatory"

            # --- proliferation ---
            if any(k in text for k in ["cell cycle", "mitosis", "dna replication"]):
                return "Proliferating cells"

            # --- metabolism ---
            if any(k in text for k in ["metabolic", "oxidative phosphorylation"]):
                return "Metabolic"

            return "Unknown"

        # -----------------------------------------------------------------------------
        # 24-3. Apply annotation
        # -----------------------------------------------------------------------------
        summary_df["suggested_label"] = summary_df.apply(
            lambda row: suggest_label(row[GO_COL], row[PW_COL]),
            axis=1
        )

        # -----------------------------------------------------------------------------
        # 24-4. Save
        # -----------------------------------------------------------------------------
        out_file = RESULTS_DIR / f"17-{DATASET_NAME}_Cluster_annotation.csv"
        summary_df.to_csv(out_file, index=False)

        print("\n" + "="*60)
        print("[OUTPUT] Annotation file saved:")
        print(out_file)
        print("="*60)

        print("\n[ANNOTATION RESULT]")
        print(summary_df[["group", "suggested_label"]])

        print("\n[STEP n5-24 COMPLETED]\n")


[STEP n5-24] Generating cluster annotations...

[INFO] Summary columns: ['group', 'GO_terms_top5', 'Pathway_terms_top5', 'n_GO_terms', 'n_Pathways']
[INFO] Using GO column: GO_terms_top5
[INFO] Using Pathway column: Pathway_terms_top5

[OUTPUT] Annotation file saved:
K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\17-260504-pbmc3k_Cluster_annotation.csv

[ANNOTATION RESULT]
  group      suggested_label
0     0  T-cell / Lymphocyte
1     1              Unknown
2     2         Inflammatory
3     3  T-cell / Lymphocyte
4     4  T-cell / Lymphocyte
5     5              Unknown

[STEP n5-24 COMPLETED]



In [51]:
# =============================================================================
# n5-25) Final Report Generator (ROBUST & POLISHED)
# =============================================================================

print("\n[STEP n5-25] Generating final report...\n")

# -----------------------------------------------------------------------------
# 25-0. Check input
# -----------------------------------------------------------------------------
try:
    summary_df
except NameError:
    print("[ERROR] summary_df not found.")
else:

    if summary_df.empty:
        print("[ERROR] summary_df is empty.")
    else:

        print(f"[INFO] Columns: {list(summary_df.columns)}")

        # -----------------------------------------------------------------------------
        # 25-1. Detect column names (🔥 FIX)
        # -----------------------------------------------------------------------------
        if "GO_terms_top5" in summary_df.columns:
            GO_COL = "GO_terms_top5"
        elif "top_GO_terms" in summary_df.columns:
            GO_COL = "top_GO_terms"
        else:
            raise ValueError("GO column not found.")

        if "Pathway_terms_top5" in summary_df.columns:
            PW_COL = "Pathway_terms_top5"
        elif "top_pathways" in summary_df.columns:
            PW_COL = "top_pathways"
        else:
            raise ValueError("Pathway column not found.")

        # -----------------------------------------------------------------------------
        # 25-2. Generate report text
        # -----------------------------------------------------------------------------
        report_lines = []

        report_lines.append("Single-cell RNA-seq Functional Analysis Report")
        report_lines.append("=" * 70)
        report_lines.append("")

        for _, row in summary_df.iterrows():

            report_lines.append(f"Cluster {row['group']}")
            report_lines.append("-" * 50)

            report_lines.append(f"Suggested identity: {row['suggested_label']}")
            report_lines.append("")

            # --- GO ---
            report_lines.append("Top GO terms:")
            go_terms = str(row[GO_COL]).split("|")
            for t in go_terms:
                report_lines.append(f"  • {t.strip()}")

            report_lines.append("")

            # --- Pathway ---
            report_lines.append("Top pathways:")
            pw_terms = str(row[PW_COL]).split("|")
            for t in pw_terms:
                report_lines.append(f"  • {t.strip()}")

            report_lines.append("")
            report_lines.append("")

        report_text = "\n".join(report_lines)

        # -----------------------------------------------------------------------------
        # 25-3. Save
        # -----------------------------------------------------------------------------
        out_txt = RESULTS_DIR / f"18-{DATASET_NAME}_Final_Report.txt"

        with open(out_txt, "w", encoding="utf-8") as f:
            f.write(report_text)

        print("\n" + "=" * 70)
        print("[OUTPUT] Final report saved:")
        print(out_txt)
        print("=" * 70)

        # -----------------------------------------------------------------------------
        # 25-4. Preview
        # -----------------------------------------------------------------------------
        print("\n[REPORT PREVIEW]\n")
        print("\n".join(report_lines[:40]))

        print("\n[STEP n5-25 COMPLETED]\n")


[STEP n5-25] Generating final report...

[INFO] Columns: ['group', 'GO_terms_top5', 'Pathway_terms_top5', 'n_GO_terms', 'n_Pathways', 'suggested_label']

[OUTPUT] Final report saved:
K:\@scRNA-1.1-PBMC3K\results\260504-pbmc3k\18-260504-pbmc3k_Final_Report.txt

[REPORT PREVIEW]

Single-cell RNA-seq Functional Analysis Report

Cluster 0
--------------------------------------------------
Suggested identity: T-cell / Lymphocyte

Top GO terms:
  • T cell activation (GO:0042110)
  • positive regulation of T cell activation (GO:0050870)
  • antigen receptor-mediated signaling pathway (GO:0050851)
  • T cell receptor signaling pathway (GO:0050852)
  • cellular response to cytokine stimulus (GO:0071345)

Top pathways:
  • Eukaryotic Translation Elongation R-HSA-156842
  • Peptide Chain Elongation R-HSA-156902
  • Viral mRNA Translation R-HSA-192823
  • Eukaryotic Translation Termination R-HSA-72764
  • Selenocysteine Synthesis R-HSA-2408557


Cluster 1
-----------------------------------------